# **Olist Marketplace Analytics Descriptive Analysis**

This notebook presents the **descriptive analysis for Chapters 1–5** of the Olist Marketplace Analytics project. All figures are generated directly from the validated BigQuery analytical layer and are intended to establish the empirical patterns that will later be tested through statistical inference and predictive modelling.

### How to read this notebook

Each figure follows the same analytical structure:

> **Question** — What business or product question are we investigating?  
> **Grain** — What does one row of the underlying analysis represent?  
> **Observation** — What does the data actually show?  
> **Implication** — What decision or hypothesis does the observation motivate, and what can we *not* conclude from it?

### Analytical discipline

This notebook is **descriptive, not causal**. The Olist dataset is observational and contains no randomized treatment or control group. Accordingly, patterns are described as **associations, differences, or trends** rather than causal effects. Formal hypothesis testing and regression are handled separately in `03_statistical_analysis.ipynb`.

### Unit of analysis

The analytical grain is matched to the question:

- **Order-level outcomes** — delivery time, lateness, review score, and order-level customer experience use `v_fact_orders`, with **one row per order**.
- **Item-level economics** — product price, freight, and freight-to-item-value analysis use `v_fact_order_items`, with **one row per order item**.
- **Customer-level analysis** — repeat purchasing and retention use `customer_unique_id`, which identifies the underlying customer rather than the order-specific `customer_id`.
- **Seller-level analysis** — seller concentration and seller performance use seller-level aggregates.

Maintaining the correct grain is essential: using item-level data for an order-level outcome would cause multi-item orders to contribute the same order-level observation multiple times, artificially inflating the sample size and potentially understating uncertainty.




---

## 1 · Setup

Installs SciencePlots and authenticates to BigQuery. Keep `USE_LATEX = False` whileiterating — the LaTeX toolchain takes several minutes to install and is only needed forfinal export, when figure fonts should match the report.

In [ ]:
!pip install SciencePlots --quiet

USE_LATEX = False  # Flip to True for final report-quality typography

if USE_LATEX:
    !apt-get install -y texlive-latex-extra texlive-fonts-recommended dvipng cm-super -qq

In [ ]:
from google.colab import auth

auth.authenticate_user()

print("Authenticated.")

Configuration and helpers. `q()` rewrites `` `olist.` `` into the full project path so theSQL below reads exactly as it does in the `sql/` directory. The smoke test at the endconfirms both authentication and dataset access before anything else runs.

In [ ]:
import warnings
import os

warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np


import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import scienceplots

from google.cloud import bigquery


# 1. CONFIGURATION

PROJECT = "blinkit-analysis-503505"
DATASET = "olist"
FIGDIR = "figures"

os.makedirs(FIGDIR, exist_ok=True)

USE_LATEX = False

STYLE = ["science", "no-latex", "grid"]

plt.style.use(STYLE)

plt.rcParams["text.usetex"] = False

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
})


# 2. COLOR PALETTE

C_MAIN = "#0C5DA5"
C_ACCENT = "#00B945"
C_WARN = "#FF2C00"
C_MUTED = "#845B97"


# 3. BIGQUERY CLIENT

client = bigquery.Client(project=PROJECT)


# 4. HELPER FUNCTIONS

def q(sql: str) -> pd.DataFrame:
    """Run a BigQuery SQL query and return a pandas DataFrame."""

    return client.query(
        sql.replace(
            "`olist.",
            f"`{PROJECT}.{DATASET}."
        )
    ).to_dataframe()


def save(fig, name: str):
    """Save a figure as both PDF and PNG."""

    fig.savefig(
        f"{FIGDIR}/{name}.pdf",
        bbox_inches="tight"
    )

    fig.savefig(
        f"{FIGDIR}/{name}.png",
        bbox_inches="tight"
    )

    print("saved:", name)


def barlabels(ax, bars, fmt="{:.0f}", dy=0.01):
    """Write value labels above bars."""

    top = ax.get_ylim()[1]

    for b in bars:
        ax.text(
            b.get_x() + b.get_width() / 2,
            b.get_height() + top * dy,
            fmt.format(b.get_height()),
            ha="center",
            va="bottom",
            fontsize=7
        )


# 5. SETUP CHECK

print("BigQuery client ready.")
print("Project:", PROJECT)
print("Dataset:", DATASET)
print("Figure directory:", FIGDIR)

# 6. SMOKE TEST


_t = q(
    "SELECT COUNT(*) AS n "
    "FROM `olist.v_orders_enriched`"
)

print(
    "v_orders_enriched rows:",
    int(_t["n"].iloc[0])
)

---

# **Chapter 1 · Marketplace Overview**

**How large is Olist, and how did the marketplace evolve over time?**  
Establishes the scale, growth trajectory, and overall structure of the marketplace before examining customer, seller, and fulfilment dynamics.

## **Figure 1 — Monthly Order Volume**

**Question.** Is Olist still in a growth phase, or has demand begun to mature? This establishes the marketplace's growth trajectory and provides context for interpreting the retention problem.

**Grain.** One row per order, aggregated by month.

**What to look for.** The overall growth pattern, changes in monthly demand, and whether the pronounced November 2017 spike represents a temporary peak or part of a broader upward trend.

**Implication.** The figure provides the temporal baseline for the analyses that follow; it describes when demand changed, but does not by itself explain why.

In [ ]:

# FIG 01 — MONTHLY ORDER GROWTH

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import FancyBboxPatch, Circle
from matplotlib.lines import Line2D



# 0. Disable LaTeX, set a clean consistent font

plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "DejaVu Sans"


# 1. Query monthly order data

df = q("""
    SELECT
        DATE_TRUNC(DATE(purchase_ts), MONTH) AS order_month,
        COUNT(*) AS orders,
        COUNT(DISTINCT customer_unique_id) AS active_customers
    FROM `olist.v_orders_enriched`
    GROUP BY order_month
    ORDER BY order_month
""")

df["order_month"] = pd.to_datetime(df["order_month"])

# Analysis period
# Trimmed: the 2016 head and Sep-Oct 2018 tail are too sparse
d = df[
    (df["order_month"] >= "2017-01-01") &
    (df["order_month"] <= "2018-08-31")
].copy()

d = d.sort_values("order_month").reset_index(drop=True)



# 2. Colour palette

GREEN = "#087F5B"
DARK_GREEN = "#075E45"
LIGHT_GREEN = "#EAF5F0"
RED = "#D62828"
BLACK = "#111111"
GREY = "#555555"
GRID = "#D8D8D8"
WHITE = "#FFFFFF"



# 3. Create figure — exact 4:3

fig, ax = plt.subplots(figsize=(12, 9), dpi=150)

# Manual spacing.
# NOTE: do NOT use bbox_inches="tight" when saving —
# it re-crops the canvas and knocks all the figure-fraction
# elements (takeaway box, icon, text) out of alignment with the axes.

fig.subplots_adjust(
    left=0.085,
    right=0.965,
    top=0.800,
    bottom=0.265
)



# 4. Title and subtitle

fig.text(
    0.5,
    0.952,
    "Order volume grew sharply through 2017, then plateaued",
    ha="center",
    va="center",
    fontsize=24,
    fontweight="bold",
    color=BLACK
)

fig.text(
    0.5,
    0.915,
    "Monthly orders from Jan 2017 to Aug 2018",
    ha="center",
    va="center",
    fontsize=15,
    fontstyle="italic",
    color=GREY
)



# 5. Main order-volume curve

ax.plot(
    d["order_month"],
    d["orders"],
    color=GREEN,
    linewidth=3.2,
    marker="o",
    markersize=9.5,
    markerfacecolor=GREEN,
    markeredgecolor=WHITE,
    markeredgewidth=2.0,
    zorder=5
)


# 6. Value labels
# Local minima are placed BELOW the point so the label never
# collides with the line. Endpoints are always placed above.


vals = d["orders"].values

for i, row in d.iterrows():

    prev_v = vals[i - 1] if i > 0 else np.inf
    next_v = vals[i + 1] if i < len(vals) - 1 else np.inf

    is_min = (
        (0 < i < len(vals) - 1)
        and vals[i] < prev_v
        and vals[i] < next_v
    )

    offset = (0, -24) if is_min else (0, 14)
    valign = "top" if is_min else "bottom"

    ax.annotate(
        f"{row['orders']:,.0f}",
        xy=(row["order_month"], row["orders"]),
        xytext=offset,
        textcoords="offset points",
        ha="center",
        va=valign,
        fontsize=11,
        color=BLACK,
        zorder=8
    )



# 7. Black Friday annotation

bf = d[d["order_month"] == pd.Timestamp("2017-11-01")]

if not bf.empty:
    ax.annotate(
        "Black Friday",
        xy=(
            bf["order_month"].iloc[0],
            bf["orders"].iloc[0]
        ),
        xytext=(-120, 52),
        textcoords="offset points",
        ha="center",
        va="center",
        fontsize=14,
        fontweight="bold",
        color=RED,
        bbox=dict(
            boxstyle="round,pad=0.55",
            facecolor=WHITE,
            edgecolor=RED,
            linewidth=2.0
        ),
        arrowprops=dict(
            arrowstyle="-|>",
            color=RED,
            linewidth=2.0,
            shrinkA=6,
            shrinkB=8,
            connectionstyle="arc3,rad=0"
        ),
        zorder=12
    )



# 8. Axis labels

ax.set_xlabel(
    "Month",
    fontsize=14,
    fontweight="bold",
    labelpad=14,
    color=BLACK
)

ax.set_ylabel(
    "Orders",
    fontsize=14,
    fontweight="bold",
    labelpad=12,
    color=BLACK
)



# 9. Y-axis — headroom so the callout sits inside the frame

ax.set_ylim(0, 9000)

ax.set_yticks(
    np.arange(0, 9001, 1000)
)

ax.set_yticklabels(
    [
        f"{x:,.0f}"
        for x in np.arange(0, 9001, 1000)
    ],
    fontsize=12,
    color=BLACK
)



# 10. X-axis

ax.set_xlim(
    pd.Timestamp("2016-12-15"),
    pd.Timestamp("2018-09-05")
)

ax.xaxis.set_major_locator(
    mdates.MonthLocator(interval=2)
)

ax.xaxis.set_major_formatter(
    mdates.DateFormatter("%Y-%m")
)

plt.setp(
    ax.get_xticklabels(),
    rotation=30,
    ha="right",
    fontsize=12,
    color=BLACK
)



# 11. Grid

ax.grid(
    True,
    axis="both",
    linestyle="--",
    linewidth=0.75,
    color=GRID,
    alpha=0.75
)

ax.set_axisbelow(True)


# 12. Spines — full frame

for side in ["left", "bottom"]:
    ax.spines[side].set_visible(True)
    ax.spines[side].set_color(BLACK)
    ax.spines[side].set_linewidth(1.3)

for side in ["top", "right"]:
    ax.spines[side].set_visible(True)
    ax.spines[side].set_color(BLACK)
    ax.spines[side].set_linewidth(1.0)



# 13. Ticks


ax.tick_params(
    axis="both",
    which="major",
    labelsize=12,
    length=6,
    width=1.1,
    colors=BLACK
)

ax.minorticks_on()

ax.tick_params(
    axis="both",
    which="minor",
    length=3,
    width=0.7,
    colors=BLACK
)


# 14. Key takeaway box — spans the same width as the axes


box_x, box_y = 0.085, 0.035
box_w, box_h = 0.880, 0.105

fig.patches.append(
    FancyBboxPatch(
        (box_x, box_y),
        box_w,
        box_h,
        boxstyle="round,pad=0.008,rounding_size=0.015",
        transform=fig.transFigure,
        facecolor=LIGHT_GREEN,
        edgecolor=GREEN,
        linewidth=1.8,
        zorder=2
    )
)


cy = box_y + box_h / 2

# 15. Takeaway icon


fig.patches.append(
    Circle(
        (0.125, cy),
        0.022,
        transform=fig.transFigure,
        facecolor=WHITE,
        edgecolor=GREEN,
        linewidth=2.0,
        zorder=5
    )
)

fig.add_artist(
    Line2D(
        [0.1155, 0.1225, 0.1290, 0.1355],
        [
            cy - 0.007,
            cy + 0.001,
            cy - 0.004,
            cy + 0.010
        ],
        transform=fig.transFigure,
        color=GREEN,
        linewidth=2.5,
        zorder=6
    )
)


# 16. Takeaway text


fig.text(
    0.163,
    cy,
    "Key takeaway:",
    ha="left",
    va="center",
    fontsize=13.5,
    fontweight="bold",
    color=DARK_GREEN,
    zorder=6
)

fig.text(
    0.283,
    cy + 0.024,
    "Steady organic growth through 2017, with a one-off Black Friday peak in November.",
    ha="left",
    va="center",
    fontsize=12.5,
    color=BLACK,
    zorder=6
)

fig.text(
    0.283,
    cy - 0.024,
    "Volume plateaued around 6,500-7,000 orders/month in 2018: a maturing marketplace.",
    ha="left",
    va="center",
    fontsize=12.5,
    color=BLACK,
    zorder=6
)


# 17. Save — no bbox_inches, so the manual layout is preserved

plt.savefig(
    f"{FIGDIR}/fig_01_monthly_growth.png",
    dpi=300,
    facecolor="white"
)

plt.savefig(
    f"{FIGDIR}/fig_01_monthly_growth.pdf",
    facecolor="white"
)


# 18. Display

plt.show()

## **Observation**

Monthly order volume rises steadily throughout 2017, from roughly **800 orders in January to 4,600 by October**, before settling into a higher but relatively stable range of **~6,200–7,300 orders per month during 2018**. November 2017 is a notable peak at **7,544 orders**, but it occurs on top of an already established upward trajectory.


- The 2017 increase reflects a **sustained growth trend**, rather than a one-off Black Friday effect; order volume had already grown approximately **5.8× before November**.
- The chart is restricted to **January 2017–August 2018**. The raw dataset contains only a small number of orders at the beginning of 2016 and a thin tail through October 2018, so including those periods would create misleading boundary effects.

## **Implication**
By 2018, Olist appears to be a **maturing marketplace**, with monthly order volume broadly plateauing after the rapid 2017 expansion. With further volume growth less evident, sustaining marketplace activity increasingly depends on **retaining existing customers and/or expanding supply** making the repeat-purchase question particularly important in the chapters that follow.

---

## **Figure 2 — Peak Demand Days**

**Question.** How concentrated is daily demand, and where might short-lived order surges create additional fulfilment pressure?

**Grain.** One row per calendar day.

**What to look for.** The largest daily order volumes and whether demand spikes are isolated events or part of broader periods of elevated activity.

**Implication.** Peak-day concentration provides context for assessing fulfilment capacity and operational resilience; the figure identifies periods of unusually high demand but does not, by itself, establish whether those peaks caused delivery or service failures.

In [ ]:

# FIG 02 — TOP 15 ORDER DAYS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import FancyBboxPatch, Circle, Ellipse
from matplotlib.lines import Line2D



# 0. Disable LaTeX, set a consistent font

plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "DejaVu Sans"



# 1. Query data

df = q("""
    SELECT
        DATE(purchase_ts) AS order_date,
        COUNT(*) AS orders
    FROM `olist.v_orders_enriched`
    GROUP BY order_date
    ORDER BY orders DESC
    LIMIT 15
""")

df["order_date"] = pd.to_datetime(df["order_date"])

# Ascending so the biggest bar sits at the top of a barh
df = df.sort_values("orders", ascending=True).reset_index(drop=True)



# 2. Colour palette — teal (normal) vs crimson (Black Friday week)

TEAL = "#0F766E"
DARK_TEAL = "#115E59"
LIGHT_TEAL = "#ECFDF5"

CRIM = "#BE123C"
DARK_CRIM = "#9F1239"
LIGHT_CRIM = "#FFF1F2"

BLACK = "#151515"
GREY = "#555555"
GRID = "#DDE3EA"
WHITE = "#FFFFFF"



# 3. Identify Black Friday

# Black Friday 2017 fell on 24 November (a Friday), 1,176 orders.
# Flagging the whole of November 2017 and taking .iloc[0] on an
# ASCENDING frame picks the SMALLEST November day (29 Nov, 323) —
# which is why the annotation used to point at rank #15.
# Match the exact date instead.

BF_DATE = pd.Timestamp("2017-11-24")

BF_WEEK_START = pd.Timestamp("2017-11-24")
BF_WEEK_END = pd.Timestamp("2017-11-30")

df["is_bf"] = df["order_date"].eq(BF_DATE)

df["is_bf_week"] = df["order_date"].between(
    BF_WEEK_START,
    BF_WEEK_END
)

bf_orders = int(
    df.loc[df["is_bf"], "orders"].iloc[0]
)

bf_index = int(
    df.index[df["is_bf"]][0]
)

runner_up = int(
    df.loc[~df["is_bf"], "orders"].max()
)



# 4. Figure — exact 4:3
# Never save with bbox_inches="tight": it re-crops the canvas and
# knocks the figure-fraction elements out of alignment.


fig, ax = plt.subplots(
    figsize=(12, 9),
    dpi=150
)

fig.subplots_adjust(
    left=0.255,     # room for rank badges + date labels
    right=0.955,
    top=0.775,
    bottom=0.245
)



# 5. Title and subtitle
# Left-aligned so they clear the KPI card


fig.text(
    0.045,
    0.950,
    "The 15 busiest order days - Black Friday dominates",
    ha="left",
    va="center",
    fontsize=24,
    fontweight="bold",
    color=BLACK
)

fig.text(
    0.045,
    0.908,
    "Daily order volume across the Olist marketplace",
    ha="left",
    va="center",
    fontsize=14.5,
    fontstyle="italic",
    color=GREY
)



# 6. Bars — Black Friday darkest, its week crimson, the rest teal


df["date_label"] = df["order_date"].dt.strftime("%d %b %Y")

colors = [
    DARK_CRIM if bf else (CRIM if wk else TEAL)
    for bf, wk in zip(
        df["is_bf"],
        df["is_bf_week"]
    )
]

bars = ax.barh(
    df["date_label"],
    df["orders"],
    color=colors,
    height=0.64,
    edgecolor="none",
    zorder=3
)

max_orders = df["orders"].max()

ax.set_xlim(
    0,
    max_orders * 1.20
)



# 7. Value labels (colour matches the bar)


for bar, value, in_week in zip(
    bars,
    df["orders"],
    df["is_bf_week"]
):
    ax.text(
        bar.get_width() + max_orders * 0.014,
        bar.get_y() + bar.get_height() / 2,
        f"{value:,}",
        ha="left",
        va="center",
        fontsize=12,
        fontweight="bold",
        color=DARK_CRIM if in_week else DARK_TEAL,
        zorder=5
    )


# 8. Rank badges

fig.text(
    0.062,
    0.795,
    "RANK",
    ha="left",
    va="center",
    fontsize=10.5,
    fontweight="bold",
    color=GREY
)

for i in range(len(df)):

    rank = len(df) - i
    is_top = (rank == 1)

    ax.add_patch(
        Ellipse(
            (-0.185, i),
            width=0.055,
            height=0.62,
            transform=ax.get_yaxis_transform(),
            clip_on=False,
            facecolor=LIGHT_CRIM if is_top else WHITE,
            edgecolor=CRIM if is_top else "#94A3B8",
            linewidth=1.4,
            zorder=4
        )
    )

    ax.text(
        -0.185,
        i,
        f"#{rank}",
        transform=ax.get_yaxis_transform(),
        ha="center",
        va="center",
        fontsize=10.5,
        fontweight="bold",
        color=DARK_CRIM if is_top else GREY,
        zorder=5,
        clip_on=False
    )



# 9. Highlight the Black Friday row

ax.axhspan(
    bf_index - 0.46,
    bf_index + 0.46,
    color=CRIM,
    alpha=0.07,
    zorder=1
)


# 10. Axes, grid, spines

ax.set_xlabel(
    "Number of orders",
    fontsize=14,
    fontweight="bold",
    labelpad=12,
    color=BLACK
)

ax.xaxis.set_major_formatter(
    mticker.FuncFormatter(
        lambda x, _: f"{x:,.0f}"
    )
)

ax.tick_params(
    axis="x",
    labelsize=11,
    length=5,
    width=1,
    colors=BLACK
)

ax.tick_params(
    axis="y",
    labelsize=12,
    length=0,
    pad=8
)

for label, bf in zip(
    ax.get_yticklabels(),
    df["is_bf"]
):
    label.set_fontweight(
        "bold" if bf else "normal"
    )

    label.set_color(
        DARK_CRIM if bf else BLACK
    )

ax.grid(
    True,
    axis="x",
    linestyle="--",
    linewidth=0.8,
    color=GRID,
    alpha=0.85,
    zorder=0
)

ax.grid(
    False,
    axis="y"
)

ax.set_axisbelow(True)

for side in ["top", "right", "left", "bottom"]:
    ax.spines[side].set_visible(True)
    ax.spines[side].set_linewidth(1.0)
    ax.spines[side].set_color("#222222")


# 11. Black Friday annotation — on the CORRECT bar

ax.annotate(
    "Black Friday\n24 Nov 2017",
    xy=(bf_orders, bf_index),
    xytext=(-150, -46),
    textcoords="offset points",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    color=DARK_CRIM,
    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor=WHITE,
        edgecolor=CRIM,
        linewidth=1.8
    ),
    arrowprops=dict(
        arrowstyle="-|>",
        color=CRIM,
        linewidth=1.9,
        shrinkA=6,
        shrinkB=8,
        connectionstyle="arc3,rad=-0.25"
    ),
    zorder=11
)



# 12. Top-right KPI card

card_x, card_y = 0.775, 0.795
card_w, card_h = 0.180, 0.120

fig.patches.append(
    FancyBboxPatch(
        (card_x, card_y),
        card_w,
        card_h,
        boxstyle="round,pad=0.008,rounding_size=0.015",
        transform=fig.transFigure,
        facecolor=LIGHT_CRIM,
        edgecolor=CRIM,
        linewidth=1.6,
        zorder=10
    )
)

fig.text(
    card_x + card_w / 2,
    card_y + card_h - 0.036,
    f"{bf_orders:,}",
    ha="center",
    va="center",
    fontsize=25,
    fontweight="bold",
    color=DARK_CRIM,
    zorder=12
)

fig.text(
    card_x + card_w / 2,
    card_y + 0.046,
    "orders on Black Friday",
    ha="center",
    va="center",
    fontsize=10.5,
    color=BLACK,
    zorder=12
)

fig.text(
    card_x + card_w / 2,
    card_y + 0.024,
    f"{bf_orders / runner_up:.1f}x the next busiest day",
    ha="center",
    va="center",
    fontsize=9.5,
    fontstyle="italic",
    color=DARK_CRIM,
    zorder=12
)



# 13. Bottom key-takeaway box

box_x, box_y = 0.085, 0.035
box_w, box_h = 0.870, 0.105

fig.patches.append(
    FancyBboxPatch(
        (box_x, box_y),
        box_w,
        box_h,
        boxstyle="round,pad=0.008,rounding_size=0.015",
        transform=fig.transFigure,
        facecolor=LIGHT_TEAL,
        edgecolor=TEAL,
        linewidth=1.7,
        zorder=2
    )
)

tcy = box_y + box_h / 2



# Takeaway icon

fig.patches.append(
    Circle(
        (0.125, tcy),
        0.022,
        transform=fig.transFigure,
        facecolor=WHITE,
        edgecolor=TEAL,
        linewidth=1.8,
        zorder=5
    )
)

fig.add_artist(
    Line2D(
        [0.1155, 0.1225, 0.1290, 0.1355],
        [
            tcy - 0.007,
            tcy + 0.001,
            tcy - 0.004,
            tcy + 0.010
        ],
        transform=fig.transFigure,
        color=TEAL,
        linewidth=2.4,
        solid_capstyle="round",
        zorder=6
    )
)



# Takeaway text


fig.text(
    0.163,
    tcy,
    "Key takeaway:",
    ha="left",
    va="center",
    fontsize=13.5,
    fontweight="bold",
    color=DARK_TEAL,
    zorder=6
)

fig.text(
    0.283,
    tcy + 0.024,
    f"Black Friday 2017 drove {bf_orders:,} orders - "
    f"{bf_orders / runner_up:.1f}x the next busiest day.",
    ha="left",
    va="center",
    fontsize=12.5,
    color=BLACK,
    zorder=6
)

fig.text(
    0.283,
    tcy - 0.024,
    "Six of the top 15 days fall in Black Friday week; "
    "the rest cluster in May and Aug 2018.",
    ha="left",
    va="center",
    fontsize=12.5,
    color=BLACK,
    zorder=6
)



# 14. Save — no bbox_inches

plt.savefig(
    f"{FIGDIR}/fig_03_top_days.png",
    dpi=300,
    facecolor="white"
)

plt.savefig(
    f"{FIGDIR}/fig_03_top_days.pdf",
    facecolor="white"
)

plt.show()

## **Observation**
Black Friday (**24 November 2017**) recorded **1,176 orders**, making it the busiest day in the dataset and **2.4× the volume of the next-busiest day** (25 November, 499 orders). Six of the 15 busiest days occurred during Black Friday week, while the remaining high-volume days were concentrated mainly in May and August 2018, with no individual day exceeding 372 orders.

## **Implication**
The **2.4× single-day demand surge** represents a substantial temporary load relative to the typical daily volume. It identifies Black Friday as a plausible period of fulfilment stress and motivates the later analysis of whether unusually high demand is associated with delivery delays or poorer customer outcomes. The figure itself, however, does not establish that the surge caused those failures.

---

# **Chapter 2  Marketplace Supply**

**Where are sellers concentrated relative to customers, and how concentrated is the marketplace's supply base?**  
Examines the geographic distribution of sellers, the balance between supply and customer demand, and the extent to which marketplace sales are concentrated among a small group of sellers.

## **Figure 3 - Supply and Demand by State**

**Question.** Is Olist's seller base geographically aligned with its customer base? A substantial mismatch could increase seller-customer distances and create additional fulfilment friction.

**Grain.** One row per state. Demand is measured by distinct customers; supply is measured by registered sellers.

**What to look for.** States where the share of customers differs materially from the share of sellers, particularly regions with relatively high demand but limited local supply.

In [ ]:

# FIG 03 — SUPPLY vs DEMAND SHARE BY STATE

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Ellipse, Rectangle


plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "DejaVu Sans"



# 1. Query

df = q("""
    WITH cust AS (
        SELECT
            customer_state AS state,
            COUNT(DISTINCT customer_unique_id) AS customers
        FROM `olist.v_orders_enriched`
        GROUP BY state
    ),
    sell AS (
        SELECT
            UPPER(TRIM(seller_state)) AS state,
            COUNT(*) AS sellers
        FROM `olist.sellers`
        GROUP BY state
    )

    SELECT
        COALESCE(c.state, s.state) AS state,
        IFNULL(c.customers, 0) AS customers,
        IFNULL(s.sellers, 0) AS sellers,

        ROUND(
            100.0 * IFNULL(c.customers, 0)
            / SUM(IFNULL(c.customers, 0)) OVER (),
            2
        ) AS pct_demand,

        ROUND(
            100.0 * IFNULL(s.sellers, 0)
            / SUM(IFNULL(s.sellers, 0)) OVER (),
            2
        ) AS pct_supply

    FROM cust c
    FULL OUTER JOIN sell s
        ON c.state = s.state

    ORDER BY customers DESC
""")


# Top 12 by demand, reversed so the biggest sits at the top
# of a barh
d = (
    df.sort_values("customers", ascending=False)
      .head(12)
      .sort_values("customers", ascending=True)
      .reset_index(drop=True)
)



# 2. Palette


DEMAND = "#173F73"
SUPPLY = "#18A999"
HL = "#E8833A"           #
HL_DARK = "#C26428"

TEXT = "#17202A"
MUTED = "#667085"
GRID = "#D9DEE5"
LIGHT_OR = "#FDF2E6"
WHITE = "#FFFFFF"



# 3. Figure
# Do NOT use the save() helper here — it applies
# bbox_inches="tight", which re-crops the canvas and pulls the
# figure-fraction takeaway box out of alignment with the axes.


FIG_W, FIG_H = 12, 7.6

fig, ax = plt.subplots(
    figsize=(FIG_W, FIG_H),
    dpi=160
)

fig.patch.set_facecolor("white")
ax.set_facecolor("white")

fig.subplots_adjust(
    left=0.095,
    right=0.965,
    top=0.815,
    bottom=0.215
)



# 4. Title and subtitle

fig.text(
    0.095,
    0.960,
    "Supply is far more concentrated in Sao Paulo than demand",
    ha="left",
    va="center",
    fontsize=21,
    fontweight="bold",
    color=TEXT
)

fig.text(
    0.095,
    0.917,
    "Share of customers vs. share of sellers across the 12 largest states",
    ha="left",
    va="center",
    fontsize=11.5,
    color=MUTED,
    style="italic"
)



# 5. Bars

y = np.arange(len(d))
bar_h = 0.34

sp_idx = int(
    d.index[d["state"] == "SP"][0]
)

# Highlight band behind the Sao Paulo row
ax.axhspan(
    sp_idx - 0.55,
    sp_idx + 0.55,
    color=LIGHT_OR,
    zorder=1
)

demand_colors = [
    HL if st == "SP" else DEMAND
    for st in d["state"]
]

supply_colors = [
    HL if st == "SP" else SUPPLY
    for st in d["state"]
]

ax.barh(
    y - bar_h / 2,
    d["pct_demand"],
    height=bar_h,
    color=demand_colors,
    zorder=3
)

ax.barh(
    y + bar_h / 2,
    d["pct_supply"],
    height=bar_h,
    color=supply_colors,
    zorder=3
)



# 6. Percentage labels

max_val = max(
    d["pct_demand"].max(),
    d["pct_supply"].max()
)

offset = max_val * 0.014

for i, row in d.iterrows():

    ax.text(
        row["pct_demand"] + offset,
        i - bar_h / 2,
        f'{row["pct_demand"]:.1f}%',
        va="center",
        ha="left",
        fontsize=9.5,
        fontweight="bold",
        color=(
            HL_DARK
            if row["state"] == "SP"
            else DEMAND
        ),
        zorder=6
    )

    ax.text(
        row["pct_supply"] + offset,
        i + bar_h / 2,
        f'{row["pct_supply"]:.1f}%',
        va="center",
        ha="left",
        fontsize=9.5,
        fontweight="bold",
        color=(
            HL_DARK
            if row["state"] == "SP"
            else SUPPLY
        ),
        zorder=6
    )



# 7. Axes, grid, spines

ax.set_yticks(y)

ax.set_yticklabels(
    d["state"],
    fontsize=11,
    fontweight="bold",
    color=TEXT
)

for label, st in zip(
    ax.get_yticklabels(),
    d["state"]
):
    if st == "SP":
        label.set_color(HL_DARK)


ax.set_xlabel(
    "Share of total (%)",
    fontsize=11.5,
    fontweight="bold",
    color=TEXT,
    labelpad=10
)

ax.set_xlim(
    0,
    max_val * 1.17
)

ax.grid(
    axis="x",
    linestyle="--",
    linewidth=0.75,
    color=GRID,
    alpha=0.8,
    zorder=0
)

ax.set_axisbelow(True)


for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)

ax.spines["bottom"].set_color("#B8C0CC")

ax.tick_params(
    axis="x",
    labelsize=10,
    colors=MUTED,
    length=0,
    pad=7
)

ax.tick_params(
    axis="y",
    length=0,
    pad=10
)



# 8. Legend


legend = ax.legend(
    handles=[
        Rectangle((0, 0), 1, 1, color=DEMAND),
        Rectangle((0, 0), 1, 1, color=SUPPLY),
        Rectangle((0, 0), 1, 1, color=HL)
    ],
    labels=[
        "Demand - customers",
        "Supply - sellers",
        "Sao Paulo"
    ],
    loc="lower right",
    bbox_to_anchor=(1.0, 1.005),
    frameon=False,
    fontsize=10,
    ncol=3,
    handlelength=1.4,
    columnspacing=1.6
)

for t in legend.get_texts():
    t.set_color(TEXT)


# 9. Key takeaway box

sp = d.loc[sp_idx]

gap = (
    sp["pct_supply"]
    - sp["pct_demand"]
)

box_x, box_y = 0.095, 0.030
box_w, box_h = 0.870, 0.105

fig.patches.append(
    FancyBboxPatch(
        (box_x, box_y),
        box_w,
        box_h,
        boxstyle="round,pad=0.010,rounding_size=0.012",
        transform=fig.transFigure,
        facecolor="#F3F7FC",
        edgecolor="#2B5DA8",
        linewidth=1.3,
        zorder=10
    )
)

tcy = box_y + box_h / 2


# Figure-fraction circles distort on a non-square figure —
# scale the height by the aspect ratio to keep it round.
AR = FIG_W / FIG_H

fig.patches.append(
    Ellipse(
        (0.135, tcy),
        width=0.030,
        height=0.030 * AR,
        transform=fig.transFigure,
        facecolor=WHITE,
        edgecolor=DEMAND,
        linewidth=1.7,
        zorder=11
    )
)


for dx, h in [
    (-0.0072, 0.016),
    (0.0, 0.026),
    (0.0072, 0.036)
]:

    fig.patches.append(
        Rectangle(
            (
                0.135 + dx - 0.0025,
                tcy - 0.020
            ),
            0.0050,
            h,
            transform=fig.transFigure,
            facecolor=DEMAND,
            edgecolor="none",
            zorder=12
        )
    )


fig.text(
    0.172,
    tcy,
    "Key takeaway:",
    fontsize=11.5,
    fontweight="bold",
    color=DEMAND,
    va="center",
    zorder=12
)

fig.text(
    0.275,
    tcy + 0.023,
    f"Sao Paulo holds {sp['pct_supply']:.1f}% of sellers but only "
    f"{sp['pct_demand']:.1f}% of customers - a {gap:.1f} pp gap.",
    fontsize=11,
    color=TEXT,
    va="center",
    zorder=12
)

fig.text(
    0.275,
    tcy - 0.023,
    "58% of demand sits outside SP while only 40% of supply does, "
    "forcing long-distance shipping.",
    fontsize=11,
    color=TEXT,
    va="center",
    zorder=12
)



# 10. Save

plt.savefig(
    f"{FIGDIR}/fig_07_supply_demand_state.png",
    dpi=300,
    facecolor="white"
)

plt.savefig(
    f"{FIGDIR}/fig_07_supply_demand_state.pdf",
    facecolor="white"
)

plt.show()

## **Observation**

São Paulo accounts for **59.7% of sellers but 42.0% of customers**, a **17.8 percentage-point gap**. More broadly, **58% of customers are located outside São Paulo, while only 40% of sellers are**, indicating that seller supply is more geographically concentrated than customer demand.

- **Paraná is even more supply-concentrated relative to demand:** 11.3% of sellers versus 5.1% of customers, giving a supply-to-demand share ratio of **2.2×**, compared with **1.4× in São Paulo**.
- **Rio de Janeiro shows the largest supply deficit:** 12.9% of customers versus only 5.5% of sellers. As the second-largest customer market, this makes RJ the clearest potential seller-recruitment opportunity in the observed data.

## **Implication**

The defensible conclusion is not that "sellers are in São Paulo while buyers are everywhere." Customers are also concentrated in São Paulo. The stronger finding is that **seller supply is more geographically concentrated than customer demand**. This structural mismatch is associated with longer shipping distances, which is examined separately in Chapter 4 rather than inferred from this figure.

## **Limitation**

This figure establishes a **geographic supply-demand imbalance**, but cannot by itself show that the imbalance causes longer delivery times or higher freight costs. Those relationships require separate order-level analysis.

---

## **Figure 4 — Seller Revenue Concentration**

**Question.** How dependent is the marketplace on a relatively small group of sellers? This informs marketplace concentration, resilience, and the potential reach of targeted seller-quality interventions.

**Grain.** One row per seller. Revenue is defined as **Σ item price**, representing GMV from item sales and excluding freight. The dataset does not contain cost-of-goods or take-rate information, so this measure should not be interpreted as profit.

**What to look for.** The cumulative share of GMV captured by the highest-value sellers and the extent to which marketplace activity is concentrated among a relatively small part of the seller base.

**Implication.** A highly concentrated seller base suggests that a relatively small number of high-value sellers may account for a disproportionate share of marketplace activity, making them important candidates for targeted operational or quality interventions.

In [ ]:

# FIG 4 — SELLER PARETO CURVE
# Presentation / Report Quality

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Ellipse, Rectangle


plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "DejaVu Sans"



# 1. Query

df = q("""
    WITH s AS (
        SELECT
            seller_id,
            SUM(price) AS revenue
        FROM `olist.v_fact_order_items`
        GROUP BY seller_id
    )

    SELECT
        ROW_NUMBER() OVER (ORDER BY revenue DESC) AS rank,
        revenue,
        SUM(revenue) OVER (
            ORDER BY revenue DESC
        ) AS cum_rev,
        SUM(revenue) OVER () AS total_rev,
        COUNT(*) OVER () AS n_sellers
    FROM s
    ORDER BY rank
""")


df["pct_sellers"] = (
    100 * df["rank"] / df["n_sellers"]
)

df["pct_revenue"] = (
    100 * df["cum_rev"] / df["total_rev"]
)

n_sellers = int(
    df["n_sellers"].iloc[0]
)


def revenue_at(pct):
    """Cumulative revenue share at a given cumulative seller share."""
    return df.loc[
        (df["pct_sellers"] - pct).abs().idxmin(),
        "pct_revenue"
    ]


p10 = revenue_at(10)
p20 = revenue_at(20)
p50 = revenue_at(50)



# 2. Palette

NAVY = "#173F73"
ORANGE = "#E8833A"
ORANGE_D = "#C26428"

TEXT = "#162B4D"
MUTED = "#6B778C"
GRID = "#D9E0E8"
WHITE = "#FFFFFF"


# 3. Figure
# Do NOT use the save() helper — bbox_inches="tight" re-crops the
# canvas and breaks the figure-fraction takeaway box.


FIG_W, FIG_H = 10.5, 7.2

fig, ax = plt.subplots(
    figsize=(FIG_W, FIG_H),
    dpi=160
)

fig.patch.set_facecolor("white")
ax.set_facecolor("white")

fig.subplots_adjust(
    left=0.105,
    right=0.965,
    top=0.760,
    bottom=0.245
)



# 4. Title and subtitle

fig.text(
    0.105,
    0.955,
    "A small group of sellers drives most of the revenue",
    ha="left",
    va="center",
    fontsize=21,
    fontweight="bold",
    color=TEXT
)

fig.text(
    0.105,
    0.908,
    f"Cumulative revenue share across all {n_sellers:,} sellers, "
    "ranked largest to smallest",
    ha="left",
    va="center",
    fontsize=11.5,
    color=MUTED,
    style="italic"
)



# 5. Equality line, shaded gap, Pareto curve

x_eq = np.linspace(0, 100, 200)

ax.plot(
    x_eq,
    x_eq,
    color="#A8B1C1",
    linestyle="--",
    linewidth=1.4,
    label="Perfect equality",
    zorder=2
)

mask20 = df["pct_sellers"] <= 20

ax.fill_between(
    df.loc[mask20, "pct_sellers"],
    df.loc[mask20, "pct_revenue"],
    df.loc[mask20, "pct_sellers"],
    color=ORANGE,
    alpha=0.18,
    zorder=1
)

ax.plot(
    df["pct_sellers"],
    df["pct_revenue"],
    color=NAVY,
    linewidth=3,
    solid_capstyle="round",
    label="Actual seller concentration",
    zorder=4
)


# 6. The 20% reference point

ax.axvline(
    20,
    color=ORANGE,
    linestyle=":",
    linewidth=1.5,
    alpha=0.9,
    zorder=3
)

ax.axhline(
    p20,
    color=ORANGE,
    linestyle=":",
    linewidth=1.2,
    alpha=0.65,
    zorder=3
)

ax.scatter(
    20,
    p20,
    s=110,
    color=ORANGE,
    edgecolor="white",
    linewidth=2,
    zorder=6
)

ax.annotate(
    f"Top 20% of sellers\ncapture {p20:.0f}% of revenue",
    xy=(20, p20),
    xytext=(40, p20 - 24),
    fontsize=11.5,
    fontweight="bold",
    color=ORANGE_D,
    ha="center",
    va="center",
    arrowprops=dict(
        arrowstyle="-|>",
        color=ORANGE,
        lw=1.8,
        shrinkA=5,
        shrinkB=8,
        connectionstyle="arc3,rad=-0.2"
    ),
    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor="#FFF4E8",
        edgecolor=ORANGE,
        linewidth=1.3
    ),
    zorder=7
)


# 7. Secondary reference points
# One number is a claim, three is a curve

for pct, label in [
    (10, f"Top 10%\n{p10:.0f}%"),
    (50, f"Top 50%\n{p50:.0f}%")
]:

    y_val = revenue_at(pct)

    ax.scatter(
        pct,
        y_val,
        s=45,
        color=NAVY,
        edgecolor="white",
        linewidth=1.5,
        zorder=6
    )

    ax.text(
        pct + 2.5,
        y_val - 6.5,
        label,
        fontsize=9,
        color=NAVY,
        fontweight="bold",
        ha="left",
        va="center",
        linespacing=1.25,
        zorder=7
    )


# Label placed inside the shaded gap, where nothing else sits
ax.text(
    7.5,
    30,
    "HIGH SELLER\nCONCENTRATION",
    fontsize=9.5,
    fontweight="bold",
    color=ORANGE_D,
    linespacing=1.3,
    ha="center",
    va="center",
    alpha=0.9
)


# 8. Axes, grid, spines, legend

ax.set_xlabel(
    "Cumulative share of sellers (%)",
    fontsize=11.5,
    fontweight="bold",
    color=TEXT,
    labelpad=10
)

ax.set_ylabel(
    "Cumulative share of revenue (%)",
    fontsize=11.5,
    fontweight="bold",
    color=TEXT,
    labelpad=10
)

ax.set_xlim(0, 100)
ax.set_ylim(0, 100)

ax.set_xticks(
    np.arange(0, 101, 20)
)

ax.set_yticks(
    np.arange(0, 101, 20)
)

ax.xaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda v, _: f"{v:.0f}%"
    )
)

ax.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda v, _: f"{v:.0f}%"
    )
)

ax.tick_params(
    axis="both",
    labelsize=10,
    colors=MUTED,
    length=0
)

ax.grid(
    axis="both",
    linestyle="--",
    linewidth=0.7,
    color=GRID,
    alpha=0.8
)

ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_color("#C7CFDA")
ax.spines["bottom"].set_color("#C7CFDA")


legend = ax.legend(
    loc="lower right",
    frameon=True,
    fontsize=9.5,
    facecolor="white",
    edgecolor=GRID,
    framealpha=0.95
)

legend.get_frame().set_linewidth(0.8)


# 9. Key takeaway box

box_x, box_y = 0.105, 0.028
box_w, box_h = 0.860, 0.108

fig.patches.append(
    FancyBboxPatch(
        (box_x, box_y),
        box_w,
        box_h,
        boxstyle="round,pad=0.010,rounding_size=0.012",
        transform=fig.transFigure,
        facecolor="#FFF4E8",
        edgecolor=ORANGE,
        linewidth=1.3,
        zorder=10
    )
)

tcy = box_y + box_h / 2

AR = FIG_W / FIG_H

fig.patches.append(
    Ellipse(
        (0.143, tcy),
        width=0.028,
        height=0.028 * AR,
        transform=fig.transFigure,
        facecolor=WHITE,
        edgecolor=ORANGE,
        linewidth=1.7,
        zorder=11
    )
)


for dx, h in [
    (-0.0066, 0.016),
    (0.0, 0.026),
    (0.0066, 0.037)
]:

    fig.patches.append(
        Rectangle(
            (
                0.143 + dx - 0.0023,
                tcy - 0.021
            ),
            0.0046,
            h,
            transform=fig.transFigure,
            facecolor=ORANGE,
            edgecolor="none",
            zorder=12
        )
    )


top20_n = int(
    round(0.20 * n_sellers)
)


fig.text(
    0.178,
    tcy,
    "Key takeaway:",
    fontsize=11.5,
    fontweight="bold",
    color=ORANGE_D,
    va="center",
    zorder=12
)

fig.text(
    0.281,
    tcy + 0.024,
    f"The top {top20_n:,} sellers (20% of {n_sellers:,}) "
    f"generate {p20:.0f}% of marketplace revenue.",
    fontsize=11.5,
    color=TEXT,
    va="center",
    zorder=12
)

fig.text(
    0.281,
    tcy - 0.024,
    "Quality issues among this group would put the majority "
    "of GMV at risk.",
    fontsize=11.5,
    color=TEXT,
    va="center",
    zorder=12
)


# 10. Save

plt.savefig(
    f"{FIGDIR}/fig_16_seller_pareto.png",
    dpi=300,
    facecolor="white"
)

plt.savefig(
    f"{FIGDIR}/fig_16_seller_pareto.pdf",
    facecolor="white"
)

plt.show()

##**Observation**

Seller concentration is substantially steeper than the conventional 80/20 pattern:

| Seller segment | Share of item value |
|---|---:|
| Top 10% (~310 sellers) | **67%** |
| Top 20% (~619 sellers) | **83%** |
| Top 50% | **97%** |

The implication is pronounced: the **bottom half of the seller base contributes only ~3% of total item value**, while a relatively small group of high-value sellers accounts for most marketplace activity.

##**Implication**

This concentration has two sides:

- **Operationally favourable:** A seller-quality programme covering roughly **600 sellers could reach ~83% of GMV**, making targeted intervention substantially more tractable than a marketplace-wide programme.
- **Structurally risky:** Heavy dependence on a small seller base creates concentration risk; disruptions or quality failures among high-value sellers could have a disproportionate effect on marketplace activity.


---

# **Chapter 3 Customer Lifecycle**

**How many customers come back, and what does repeat purchasing look like over time?**  
The central commercial question of the project: understanding repeat behaviour, retention, and the characteristics of customers who return.

## **Figure 5 — Orders per Customer**

**Question.** What does the customer repeat-purchase distribution actually look like?

**Grain.** One row per person, identified by `customer_unique_id`, rather than the order-specific `customer_id`.

**Data-integrity note.** Olist contains two customer identifiers with different meanings. `customer_id` is assigned at the order level, so all **99,441 orders have distinct `customer_id` values**. `customer_unique_id`, by contrast, identifies the underlying person, giving **96,096 unique customers**. Using `customer_id` for retention analysis would therefore produce a misleading **0.00% repeat rate by construction**.

The **3,345-customer difference between the two identifiers represents customers associated with more than one order**, making `customer_unique_id` the correct key for all repeat-purchase and retention analysis.

In [ ]:

# FIG 5 — ORDERS PER CUSTOMER
# Presentation / Report Quality


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import FancyBboxPatch, Ellipse, Rectangle


plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "DejaVu Sans"


# 1. Query

df = q("""
    WITH opp AS (
        SELECT
            customer_unique_id,
            COUNT(DISTINCT order_id) AS n
        FROM `olist.v_orders_enriched`
        GROUP BY customer_unique_id
    )

    SELECT
        n AS order_count,
        COUNT(*) AS customers
    FROM opp
    GROUP BY n
    ORDER BY n
""")


# Percentages must use the FULL customer base, not just the <=7 subset,
# otherwise the shares are subtly wrong.
total_customers = df["customers"].sum()

d = df[
    df["order_count"] <= 7
].copy()

one_order = int(
    d.loc[
        d["order_count"] == 1,
        "customers"
    ].iloc[0]
)

one_order_pct = (
    100 * one_order / total_customers
)

repeat_pct = (
    100 - one_order_pct
)

two_order = int(
    d.loc[
        d["order_count"] == 2,
        "customers"
    ].iloc[0]
)

drop_x = (
    one_order / two_order
)


# 2. Palette


C_NAVY = "#173F73"
C_OR = "#E8833A"
C_OR_D = "#C26428"
C_TEXT = "#172033"
C_MUTED = "#667085"
C_GRID = "#D9E1EA"
C_BOX = "#F4F7FB"
WHITE = "#FFFFFF"


# 3. Figure
# Do NOT use the save() helper — it applies bbox_inches="tight",
# which re-crops the canvas and pushes the takeaway text off-figure.

FIG_W, FIG_H = 11, 6.6

fig, ax = plt.subplots(
    figsize=(FIG_W, FIG_H),
    dpi=160
)

fig.patch.set_facecolor("white")
ax.set_facecolor("white")

fig.subplots_adjust(
    left=0.105,
    right=0.965,
    top=0.775,
    bottom=0.245
)


# 4. Title and subtitle

fig.text(
    0.105,
    0.955,
    "Customer repeat ordering is extremely limited",
    ha="left",
    va="center",
    fontsize=21,
    fontweight="bold",
    color=C_TEXT
)

fig.text(
    0.105,
    0.908,
    "Distribution of customers by number of orders placed",
    ha="left",
    va="center",
    fontsize=12,
    style="italic",
    color=C_MUTED
)


# 5. Bars

colors = [
    C_OR if n == 1 else C_NAVY
    for n in d["order_count"]
]

bars = ax.bar(
    d["order_count"].astype(str),
    d["customers"],
    color=colors,
    width=0.62,
    edgecolor="white",
    linewidth=1.1,
    zorder=3
)

ax.set_yscale("log")

ax.set_ylim(
    1,
    one_order * 7
)


# 6. Value labels — count AND share

for bar, value in zip(
    bars,
    d["customers"]
):

    pct = (
        100 * value / total_customers
    )

    label = (
        f"{int(value):,}\n({pct:.2f}%)"
        if pct >= 0.01
        else f"{int(value):,}"
    )

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value * 1.30,
        label,
        ha="center",
        va="bottom",
        fontsize=9.5,
        fontweight="bold",
        color=(
            C_OR_D
            if value == one_order
            else C_TEXT
        ),
        linespacing=1.35
    )


# 7. Callout on the one-order bar

ax.annotate(
    f"{one_order_pct:.1f}% of all customers\n"
    "order exactly once",
    xy=(0, one_order),
    xytext=(1.35, one_order * 0.42),
    ha="left",
    va="center",
    fontsize=11.5,
    fontweight="bold",
    color=C_OR_D,
    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor=WHITE,
        edgecolor=C_OR,
        linewidth=1.6
    ),
    arrowprops=dict(
        arrowstyle="-|>",
        color=C_OR,
        lw=1.9,
        shrinkA=4,
        shrinkB=8,
        connectionstyle="arc3,rad=-0.2"
    ),
    zorder=10
)


# 8. Grid, ticks, spines
# minorticks_off() kills the stray dashes a log axis draws
# along the top and right edges.

ax.grid(
    axis="y",
    which="major",
    linestyle="--",
    linewidth=0.8,
    color=C_GRID,
    alpha=0.9,
    zorder=0
)

ax.grid(
    axis="y",
    which="minor",
    visible=False
)

ax.grid(
    axis="x",
    visible=False
)

ax.minorticks_off()

ax.set_axisbelow(True)


# Plain numbers instead of 10^5 notation
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(
        lambda v, _: f"{v:,.0f}"
        if v >= 1
        else ""
    )
)

ax.set_xlabel(
    "Number of orders placed",
    fontsize=12,
    fontweight="bold",
    color=C_TEXT,
    labelpad=10
)

ax.set_ylabel(
    "Customers (log scale)",
    fontsize=12,
    fontweight="bold",
    color=C_TEXT,
    labelpad=10
)

ax.tick_params(
    axis="both",
    labelsize=10.5,
    colors=C_TEXT,
    length=0
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_color("#B8C2CF")
ax.spines["bottom"].set_color("#B8C2CF")


# 9. Key takeaway box

box_x, box_y = 0.105, 0.030
box_w, box_h = 0.860, 0.115

fig.patches.append(
    FancyBboxPatch(
        (box_x, box_y),
        box_w,
        box_h,
        boxstyle="round,pad=0.010,rounding_size=0.012",
        transform=fig.transFigure,
        facecolor=C_BOX,
        edgecolor=C_NAVY,
        linewidth=1.3,
        zorder=10
    )
)

tcy = box_y + box_h / 2

AR = FIG_W / FIG_H
# Keeps the icon circle actually circular

fig.patches.append(
    Ellipse(
        (0.145, tcy),
        width=0.030,
        height=0.030 * AR,
        transform=fig.transFigure,
        facecolor=WHITE,
        edgecolor=C_NAVY,
        linewidth=1.7,
        zorder=11
    )
)


for dx, h in [
    (-0.0072, 0.038),
    (0.0, 0.022),
    (0.0072, 0.013)
]:

    fig.patches.append(
        Rectangle(
            (
                0.145 + dx - 0.0025,
                tcy - 0.022
            ),
            0.0050,
            h,
            transform=fig.transFigure,
            facecolor=C_NAVY,
            edgecolor="none",
            zorder=12
        )
    )


fig.text(
    0.182,
    tcy,
    "Key takeaway:",
    fontsize=11.5,
    fontweight="bold",
    color=C_NAVY,
    va="center",
    zorder=12
)

fig.text(
    0.285,
    tcy + 0.025,
    f"{one_order_pct:.1f}% of customers order exactly once; "
    f"only {repeat_pct:.1f}% ever return.",
    fontsize=11.5,
    color=C_TEXT,
    va="center",
    zorder=12
)

fig.text(
    0.285,
    tcy - 0.025,
    f"The drop from 1 to 2 orders is a {drop_x:.0f}x fall - "
    "repeat purchase is near-absent, not merely weak.",
    fontsize=11.5,
    color=C_TEXT,
    va="center",
    zorder=12
)


# 10. Save

plt.savefig(
    f"{FIGDIR}/fig_09_orders_per_person.png",
    dpi=300,
    facecolor="white"
)

plt.savefig(
    f"{FIGDIR}/fig_09_orders_per_person.pdf",
    facecolor="white"
)

plt.show()

## **Observation**

**93,099 of 96,096 customers (96.9%) placed exactly one order.** The distribution drops sharply after the first purchase: only **2,745 customers** placed a second order, a **34× reduction**, while the long tail is extremely thin just 203 customers reached three orders and 30 reached four.

## **Implication.**

The observed purchase distribution indicates that repeat ordering is rare in this marketplace. A simple lifetime-value model based on the raw order counts would therefore be dominated by the first purchase, with limited evidence of sustained customer frequency or lifespan.

## **Important limitation - right censoring.**

This figure overstates the apparent retention problem because customers have unequal opportunities to return. The dataset ends on **17 October 2018**, so a customer acquired in October 2018 has only days of observable follow-up, while a customer acquired in January 2017 has roughly 21 months.

To address this, repeat-purchase analysis is treated as **censoring-aware**: 30-, 60-, 90-, and 180-day repeat rates are calculated only for customers with sufficient observation time. The underlying repeat indicators use three states—**TRUE, FALSE, and NULL**—with `NULL` representing insufficient exposure rather than a failed repeat.

Accordingly, the **3.1% ever-repeated figure is retained only as a descriptive lower-bound benchmark and is not interpreted as the marketplace's retention rate.**

# **Chapter 4 Fulfilment Economics**

**What does geographic distance cost Olist in delivery time and freight?**  
Examines how seller-customer distance translates into fulfilment burden, quantifying its relationship with delivery speed and shipping cost before assessing the downstream impact on customer experience.

## **Figure 6 — Distance, Delivery Time and Freight**

**Question.** How much does seller–customer distance add to fulfilment burden in both delivery time and freight and does the relationship extend to customer satisfaction?

**Grain.** One row per order, grouped into distance bands. Delivery time and review score are measured at the order level, while freight is summed across the order's items.

**Distance definition.** For multi-item orders, distance is defined as the **maximum seller–customer distance across all items**. This reflects the customer experience more closely: an order is operationally complete only when its furthest item has arrived.

In [ ]:

# FIG 6 — DISTANCE EFFECTS (small multiples)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Ellipse, Rectangle


plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "DejaVu Sans"


# 1. Query

df = q("""
    SELECT
        CASE
            WHEN max_distance_km <   50 THEN '0-50'
            WHEN max_distance_km <  200 THEN '50-200'
            WHEN max_distance_km <  500 THEN '200-500'
            WHEN max_distance_km < 1000 THEN '500-1000'
            ELSE '1000+'
        END AS bucket,

        MIN(max_distance_km) AS srt,
        COUNT(*) AS orders,
        ROUND(AVG(delivery_days), 2) AS avg_delivery_days,
        ROUND(AVG(freight_value), 2) AS avg_freight,
        ROUND(AVG(review_score), 3) AS avg_review_score

    FROM `olist.v_fact_orders`

    WHERE order_status = 'delivered'
      AND max_distance_km IS NOT NULL

    GROUP BY bucket
    ORDER BY srt
""")


# 2. Palette

C_MAIN = "#173F73"
C_ACC = "#19A68C"
C_WARN = "#E8833A"
C_WARN_D = "#C26428"
C_TEXT = "#172033"
C_MUTED = "#667085"
C_GRID = "#D9E1EA"
WHITE = "#FFFFFF"


# 3. Figure

# sharex=True is dropped: each panel needs its own visible tick
# labels, and sharing them hides two of the three sets.
# Do NOT use the save() helper (bbox_inches="tight" breaks the
# figure-fraction takeaway box).

FIG_W, FIG_H = 13, 6.4

fig, axes = plt.subplots(
    1,
    3,
    figsize=(FIG_W, FIG_H),
    dpi=160
)

fig.patch.set_facecolor("white")

for a in axes:
    a.set_facecolor("white")

fig.subplots_adjust(
    left=0.062,
    right=0.982,
    top=0.745,
    bottom=0.265,
    wspace=0.30
)

x = np.arange(len(df))
labels = df["bucket"].tolist()


# 4. Title and subtitle

fig.text(
    0.062,
    0.955,
    "Distance raises cost and wait time - and drags satisfaction down",
    ha="left",
    va="center",
    fontsize=20,
    fontweight="bold",
    color=C_TEXT
)

fig.text(
    0.062,
    0.908,
    "Longer seller-to-customer distances mean slower delivery, "
    "higher freight and lower review scores",
    ha="left",
    va="center",
    fontsize=11.5,
    fontstyle="italic",
    color=C_MUTED
)


# 5. Shared panel formatting

def style_panel(ax, title, ylabel):

    ax.set_title(
        title,
        loc="left",
        fontsize=13.5,
        fontweight="bold",
        color=C_TEXT,
        pad=10
    )

    ax.set_ylabel(
        ylabel,
        fontsize=10.5,
        fontweight="bold",
        color=C_TEXT,
        labelpad=8
    )

    ax.set_xticks(x)

    ax.set_xticklabels(
        labels,
        fontsize=9.5,
        rotation=20,
        ha="right"
    )

    ax.grid(
        axis="y",
        linestyle="--",
        linewidth=0.8,
        color=C_GRID,
        alpha=0.9,
        zorder=0
    )

    ax.grid(
        axis="x",
        visible=False
    )

    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_color("#C5CED8")
    ax.spines["bottom"].set_color("#C5CED8")

    ax.tick_params(
        axis="both",
        labelsize=9.5,
        colors=C_TEXT,
        length=0
    )


# 6. PANEL 1 — delivery time

ax = axes[0]

bars = ax.bar(
    x,
    df["avg_delivery_days"],
    width=0.62,
    color=C_MAIN,
    zorder=3
)

ax.set_ylim(
    0,
    df["avg_delivery_days"].max() * 1.32
)

for bar, value in zip(
    bars,
    df["avg_delivery_days"]
):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + df["avg_delivery_days"].max() * 0.03,
        f"{value:.1f}d",
        ha="center",
        va="bottom",
        fontsize=9.5,
        fontweight="bold",
        color=C_TEXT
    )

style_panel(
    ax,
    "Delivery time increases",
    "Average delivery time (days)"
)

delta_days = (
    df["avg_delivery_days"].iloc[-1]
    - df["avg_delivery_days"].iloc[0]
)

mult_days = (
    df["avg_delivery_days"].iloc[-1]
    / df["avg_delivery_days"].iloc[0]
)

ax.annotate(
    f"+{delta_days:.1f} days\n({mult_days:.1f}x)",
    xy=(
        x[-1],
        df["avg_delivery_days"].iloc[-1]
    ),
    xytext=(
        x[1] - 0.1,
        df["avg_delivery_days"].max() * 1.16
    ),
    fontsize=9.5,
    fontweight="bold",
    color=C_MAIN,
    ha="center",
    va="center",
    bbox=dict(
        boxstyle="round,pad=0.4",
        facecolor="#EEF3F8",
        edgecolor=C_MAIN,
        linewidth=1.2
    ),
    arrowprops=dict(
        arrowstyle="-|>",
        color=C_MAIN,
        lw=1.4,
        shrinkA=4,
        shrinkB=6
    )
)


# 7. PANEL 2 — freight
# NOTE: escape the currency symbol as R\$ — matplotlib reads
# a pair of unescaped $ as mathtext and italicises the string.

ax = axes[1]

bars = ax.bar(
    x,
    df["avg_freight"],
    width=0.62,
    color=C_ACC,
    zorder=3
)

ax.set_ylim(
    0,
    df["avg_freight"].max() * 1.32
)

for bar, value in zip(
    bars,
    df["avg_freight"]
):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + df["avg_freight"].max() * 0.03,
        f"R\\${value:.0f}",
        ha="center",
        va="bottom",
        fontsize=9.5,
        fontweight="bold",
        color=C_TEXT
    )

style_panel(
    ax,
    "Freight cost rises",
    "Average freight (R\\$)"
)

pct_freight = (
    df["avg_freight"].iloc[-1]
    / df["avg_freight"].iloc[0]
    - 1
) * 100

ax.annotate(
    f"+{pct_freight:.0f}%",
    xy=(
        x[-1],
        df["avg_freight"].iloc[-1]
    ),
    xytext=(
        x[1] - 0.1,
        df["avg_freight"].max() * 1.16
    ),
    fontsize=10.5,
    fontweight="bold",
    color=C_ACC,
    ha="center",
    va="center",
    bbox=dict(
        boxstyle="round,pad=0.4",
        facecolor="#EAF7F4",
        edgecolor=C_ACC,
        linewidth=1.2
    ),
    arrowprops=dict(
        arrowstyle="-|>",
        color=C_ACC,
        lw=1.4,
        shrinkA=4,
        shrinkB=6
    )
)


# 8. PANEL 3 — review score

ax = axes[2]

ax.plot(
    x,
    df["avg_review_score"],
    color=C_WARN,
    linewidth=2.8,
    marker="o",
    markersize=7,
    markerfacecolor="white",
    markeredgewidth=2,
    markeredgecolor=C_WARN,
    zorder=5
)


# Symmetric padding around the actual range — a hard-coded offset
# clips the labels because this series spans only ~0.26 points.

lo = df["avg_review_score"].min()
hi = df["avg_review_score"].max()

pad = (hi - lo) * 0.9

ax.set_ylim(
    lo - pad,
    hi + pad * 1.35
)

ax.fill_between(
    x,
    df["avg_review_score"],
    lo - pad,
    color=C_WARN,
    alpha=0.10,
    zorder=1
)

for xi, value in zip(
    x,
    df["avg_review_score"]
):

    ax.text(
        xi,
        value + pad * 0.16,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9.5,
        fontweight="bold",
        color=C_TEXT
    )

style_panel(
    ax,
    "Satisfaction declines",
    "Average review score"
)

decline = (
    df["avg_review_score"].iloc[-1]
    - df["avg_review_score"].iloc[0]
)

ax.annotate(
    f"{decline:+.2f} pts",
    xy=(
        x[-1],
        df["avg_review_score"].iloc[-1]
    ),
    xytext=(
        x[1] - 0.1,
        lo - pad * 0.55
    ),
    fontsize=10.5,
    fontweight="bold",
    color=C_WARN_D,
    ha="center",
    va="center",
    bbox=dict(
        boxstyle="round,pad=0.4",
        facecolor="#FFF7ED",
        edgecolor=C_WARN,
        linewidth=1.2
    ),
    arrowprops=dict(
        arrowstyle="-|>",
        color=C_WARN,
        lw=1.4,
        shrinkA=4,
        shrinkB=6
    )
)


# ------------------------------------------------------------
# 9. Shared x-axis label
# ------------------------------------------------------------

fig.text(
    0.5,
    0.172,
    "Seller to customer distance (km)",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    color=C_TEXT
)



# 10. Key takeaway box

box_x, box_y = 0.062, 0.028
box_w, box_h = 0.920, 0.105

fig.patches.append(
    FancyBboxPatch(
        (box_x, box_y),
        box_w,
        box_h,
        boxstyle="round,pad=0.010,rounding_size=0.012",
        transform=fig.transFigure,
        facecolor="#F4F7FB",
        edgecolor=C_MAIN,
        linewidth=1.3,
        zorder=10
    )
)

tcy = box_y + box_h / 2

AR = FIG_W / FIG_H

fig.patches.append(
    Ellipse(
        (0.100, tcy),
        width=0.026,
        height=0.026 * AR,
        transform=fig.transFigure,
        facecolor=WHITE,
        edgecolor=C_MAIN,
        linewidth=1.7,
        zorder=11
    )
)


for dx, h in [
    (-0.0062, 0.015),
    (0.0, 0.025),
    (0.0062, 0.036)
]:

    fig.patches.append(
        Rectangle(
            (
                0.100 + dx - 0.0022,
                tcy - 0.020
            ),
            0.0044,
            h,
            transform=fig.transFigure,
            facecolor=C_MAIN,
            edgecolor="none",
            zorder=12
        )
    )


fig.text(
    0.132,
    tcy,
    "Key takeaway:",
    fontsize=11.5,
    fontweight="bold",
    color=C_MAIN,
    va="center",
    zorder=12
)

fig.text(
    0.233,
    tcy + 0.024,
    f"Beyond 1,000 km, delivery takes "
    f"{df['avg_delivery_days'].iloc[-1]:.1f} days "
    f"({delta_days:+.1f} vs local) and freight costs "
    f"{pct_freight:.0f}% more.",
    fontsize=11.5,
    color=C_TEXT,
    va="center",
    zorder=12
)

fig.text(
    0.233,
    tcy - 0.024,
    f"Review scores fall only {abs(decline):.2f} pts, "
    "so distance hurts economics far more than satisfaction.",
    fontsize=11.5,
    color=C_TEXT,
    va="center",
    zorder=12
)


# 11. Save

plt.savefig(
    f"{FIGDIR}/fig_14_distance_effects.png",
    dpi=300,
    facecolor="white"
)

plt.savefig(
    f"{FIGDIR}/fig_14_distance_effects.pdf",
    facecolor="white"
)

plt.show()

## **Observation**
The distance gradient is pronounced: moving from orders under 50 km to orders over 1,000 km, **delivery time increases from 5.7 to 18.7 days (3.3×)** and **freight rises from R\$13 to R$36 (+170%)**. In contrast, average review score falls by only **0.25 points**, from **4.28 to 4.02**.

**The key pattern is the asymmetry.** Distance has a large operational and economic effect, but a comparatively modest unconditional relationship with satisfaction.

The controlled analysis provides further context: once delivery time is included in the regression, the coefficient on log-distance becomes **positive**, suggesting that customers respond more to the time they wait than to geographic distance itself. Long-distance orders may also carry longer promised delivery windows, which can shape customer expectations.

## **Implication**

Distance appears to be primarily a **fulfilment-cost and efficiency problem rather than a direct satisfaction problem**. This shifts the product focus toward seller placement, network design, and freight strategy rather than treating geographic distance alone as a customer-experience issue.

## **Limitation**

Distance bands differ in composition, including category mix, order value, and seller characteristics. The band comparison is therefore descriptive. The controlled regression provides a stronger estimate of the relationship and should be used for the formal statistical conclusion in the next chapter.

---

## **Figure 7 — Shipping Cost Burden by Price Band**

**Question.** At what item price does freight become a materially smaller share of the value being shipped?

**Grain.** One row per **order item**. Item-level grain is appropriate because the analysis compares product price with the freight charged for that specific item.

**Terminology.** The metric is **freight as a share of item value**:

$$
\text{Freight Burden}
=
\frac{\text{Freight Value}}{\text{Item Price}}
\times 100
$$



This measures shipping-cost burden, **not margin or profitability**. The dataset does not contain cost of goods, seller margins, or Olist's take rate, so the analysis cannot support claims about whether an item or seller is profitable.


In [ ]:

# FIG 7 — FREIGHT ECONOMICS BY ITEM PRICE BAND

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Ellipse, Rectangle


plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "DejaVu Sans"



# 1. Query
# Band labels use R\$ — an unescaped $ pair turns the string
# into italic mathtext.

df = q("""
    SELECT
        CASE
            WHEN price <  30 THEN 'R$0-30'
            WHEN price <  60 THEN 'R$30-60'
            WHEN price < 120 THEN 'R$60-120'
            WHEN price < 300 THEN 'R$120-300'
            ELSE 'R$300+'
        END AS band,

        MIN(price) AS srt,
        COUNT(*) AS items,
        ROUND(AVG(price), 2) AS avg_price,
        ROUND(AVG(freight_value), 2) AS avg_freight,
        ROUND(
            100.0 * AVG(
                freight_value / NULLIF(price, 0)
            ),
            1
        ) AS freight_pct

    FROM `olist.v_fact_order_items`

    WHERE price > 0

    GROUP BY band
    ORDER BY srt
""")


for col in [
    "avg_price",
    "avg_freight",
    "freight_pct"
]:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )


# Escape the $ HERE, for matplotlib only — an unescaped $
# pair is read as mathtext and italicises the label.

df["band"] = df["band"].str.replace(
    "$",
    r"\$",
    regex=False
)


# 2. Palette

NAVY = "#17365D"
TEAL = "#18A999"
ORANGE = "#E8833A"
RED = "#D6452B"
GRID = "#DCE3EA"
TEXT = "#172B4D"
MUTED = "#6B778C"
WHITE = "#FFFFFF"


# Colour derived from the value, not hard-coded by position —
# a fixed 5-colour list breaks if the bands ever change.
def band_color(pct):

    if pct >= 60:
        return RED

    if pct >= 30:
        return ORANGE

    if pct >= 15:
        return TEAL

    return NAVY



# 3. Figure
# Do NOT use the save() helper — bbox_inches="tight" re-crops
# the canvas and breaks the figure-fraction takeaway box.


FIG_W, FIG_H = 11.5, 7.0

fig, ax = plt.subplots(
    figsize=(FIG_W, FIG_H),
    dpi=160
)

fig.patch.set_facecolor("white")
ax.set_facecolor("white")

fig.subplots_adjust(
    left=0.085,
    right=0.965,
    top=0.745,
    bottom=0.315
)

x = np.arange(len(df))



# 4. Title and subtitle

fig.text(
    0.085,
    0.955,
    "Cheap items carry the heaviest freight burden",
    ha="left",
    va="center",
    fontsize=21,
    fontweight="bold",
    color=TEXT
)

fig.text(
    0.085,
    0.908,
    "Freight becomes a disproportionately large share of "
    "product value as item prices fall",
    ha="left",
    va="center",
    fontsize=11.5,
    color=MUTED,
    style="italic"
)


# 5. Bars and value labels

colors = [
    band_color(p)
    for p in df["freight_pct"]
]

bars = ax.bar(
    x,
    df["freight_pct"],
    width=0.62,
    color=colors,
    zorder=3
)

ax.set_ylim(
    0,
    df["freight_pct"].max() * 1.55
)


for i, (bar, pct) in enumerate(
    zip(bars, df["freight_pct"])
):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        pct + df["freight_pct"].max() * 0.035,
        f"{pct:.0f}%",
        ha="center",
        va="bottom",
        fontsize=14,
        fontweight="bold",
        color=colors[i],
        zorder=5
    )


# 6. Per-band detail rows below the axis
# Placed in axes-transform coordinates, so they can never be
# clipped the way a negative data-coordinate y value would be.

for i, row in df.iterrows():

    ax.text(
        i,
        -0.085,
        f"Avg item   R\\${row['avg_price']:,.0f}",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=9.5,
        color=TEXT
    )

    ax.text(
        i,
        -0.145,
        f"Freight     R\\${row['avg_freight']:,.0f}",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=9.5,
        color=MUTED
    )

    ax.text(
        i,
        -0.205,
        f"{row['items']:,} items",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=8.5,
        color=MUTED,
        fontstyle="italic"
    )



# 7. Callout on the worst band

ax.annotate(
    "Shipping absorbs three-quarters\n"
    "of the item's value",
    xy=(
        0,
        df["freight_pct"].iloc[0]
    ),
    xytext=(
        1.15,
        df["freight_pct"].max() * 1.34
    ),
    fontsize=11,
    fontweight="bold",
    color=RED,
    ha="center",
    va="center",
    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor="#FDECE8",
        edgecolor=RED,
        linewidth=1.4
    ),
    arrowprops=dict(
        arrowstyle="-|>",
        color=RED,
        lw=1.7,
        shrinkA=5,
        shrinkB=8,
        connectionstyle="arc3,rad=0.18"
    ),
    zorder=10
)



# 8. Axes, grid, spines

ax.set_xticks(x)

ax.set_xticklabels(
    df["band"],
    fontsize=11,
    fontweight="bold"
)

ax.set_ylabel(
    "Freight as % of item price",
    fontsize=12,
    fontweight="bold",
    color=TEXT,
    labelpad=10
)

ax.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda v, _: f"{v:.0f}%"
    )
)

ax.grid(
    axis="y",
    color=GRID,
    linestyle="--",
    linewidth=0.8,
    alpha=0.9,
    zorder=0
)

ax.grid(
    axis="x",
    visible=False
)

ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_color("#B8C4D1")
ax.spines["bottom"].set_color("#B8C4D1")

ax.tick_params(
    axis="y",
    colors=MUTED,
    labelsize=10,
    length=0
)

ax.tick_params(
    axis="x",
    colors=TEXT,
    length=0,
    pad=8
)


# Placed manually: a large labelpad would push it into
# the takeaway box.
fig.text(
    0.525,
    0.178,
    "Item price band",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    color=TEXT
)


# 9. Key takeaway box

box_x, box_y = 0.085, 0.028
box_w, box_h = 0.880, 0.108

fig.patches.append(
    FancyBboxPatch(
        (box_x, box_y),
        box_w,
        box_h,
        boxstyle="round,pad=0.010,rounding_size=0.012",
        transform=fig.transFigure,
        facecolor="#FDECE8",
        edgecolor=RED,
        linewidth=1.3,
        zorder=10
    )
)

tcy = box_y + box_h / 2

AR = FIG_W / FIG_H


fig.patches.append(
    Ellipse(
        (0.122, tcy),
        width=0.026,
        height=0.026 * AR,
        transform=fig.transFigure,
        facecolor=WHITE,
        edgecolor=RED,
        linewidth=1.7,
        zorder=11
    )
)


for dx, h in [
    (-0.0062, 0.036),
    (0.0, 0.024),
    (0.0062, 0.014)
]:

    fig.patches.append(
        Rectangle(
            (
                0.122 + dx - 0.0022,
                tcy - 0.020
            ),
            0.0044,
            h,
            transform=fig.transFigure,
            facecolor=RED,
            edgecolor="none",
            zorder=12
        )
    )


fig.text(
    0.156,
    tcy,
    "Key takeaway:",
    fontsize=11.5,
    fontweight="bold",
    color=RED,
    va="center",
    zorder=12
)

fig.text(
    0.259,
    tcy + 0.024,
    f"Under R\\$30, freight averages "
    f"{df['freight_pct'].iloc[0]:.0f}% of item value "
    f"vs just {df['freight_pct'].iloc[-1]:.0f}% above R\\$300.",
    fontsize=11.5,
    color=TEXT,
    va="center",
    zorder=12
)

fig.text(
    0.259,
    tcy - 0.024,
    f"These {df['items'].iloc[0]:,} low-value items are "
    "structurally unprofitable to ship at distance.",
    fontsize=11.5,
    color=TEXT,
    va="center",
    zorder=12
)


# 10. Save

plt.savefig(
    f"{FIGDIR}/fig_15_freight_economics.png",
    dpi=300,
    facecolor="white"
)

plt.savefig(
    f"{FIGDIR}/fig_15_freight_economics.pdf",
    facecolor="white"
)

plt.show()

## **Observation**

Freight burden falls sharply as item price increases. Average freight rises gradually from **R\$14** for items priced below R\$30 to **R\$40** for items above R\$300, while freight as a share of item value falls from **75% to 7%**. The decline is steepest across the first three price bands and becomes more gradual above R\$120.

| Price band | Avg. item price | Avg. freight | Freight / value | Items |
|---|---:|---:|---:|---:|
| R\$0-30 | R\$21 | R\$14 | **75%** | 20,093 |
| R\$30-60 | R\$47 | R\$16 | 35% | 28,413 |
| R\$60-120 | R\$89 | R\$19 | 22% | 31,433 |
| R\$120-300 | R\$179 | R\$25 | 15% | 25,749 |
| R\$300+ | R\$634 | R\$40 | **7%** | 6,962 |

The sub-R\$30 segment is economically meaningful rather than a small edge case: **20,093 items, roughly 19% of all items sold**, fall into this band.


## **Implication**

Low-value items carry a disproportionately high shipping-cost burden. For roughly one-fifth of items, average freight is equivalent to around three-quarters of item value. This makes the segment relevant to **freight-subsidy policy, minimum-basket thresholds, assortment decisions, and the economics of serving customers over longer distances**.



## **Methodology**

The headline percentages above use the **mean of per-item freight-to-value ratios**, which can be heavily influenced by very low-priced items. For example, a R5 item with R15 freight contributes a 300% ratio. A more conservative alternative is the pooled ratio:

$$\text{Pooled Freight Burden} = \frac{\sum \text{Freight Value}}{\sum \text{Item Price}} \times 100$$

Both measures should be reported. If the qualitative pattern remains consistent, the conclusion is more robust; however, the headline percentage should be based on whichever definition is chosen explicitly for the analysis.

# **Chapter 5 Customer Experience**

**How does delivery performance relate to customer review outcomes?**  
Examines the relationship between fulfilment performance and customer satisfaction, with delivery lateness emerging as the strongest observed association in the dataset.

## **Figure 8 — Review Score Distribution**

**Question.** What is the baseline distribution of customer satisfaction? Establishing the overall review profile provides the reference point for interpreting differences across delivery and fulfilment groups.

**Grain.** One row per order. Reviews are deduplicated to the most recent review for each order; **547 orders have more than one review record**.

**What to look for.** The concentration of scores across the 1–5 scale, particularly the relative prevalence of highly positive versus poor reviews, before examining how review outcomes vary with delivery performance.

In [ ]:

# FIG 18 — REVIEW SCORE DISTRIBUTION


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import FancyBboxPatch, Ellipse, Rectangle


plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "DejaVu Sans"


# 1. Query


df = q("""
    SELECT
        review_score,
        COUNT(*) AS reviews,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM `olist.v_reviews_clean`
    GROUP BY review_score
    ORDER BY review_score
""")

for col in ["review_score", "reviews", "pct"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")


# 2. Palette — a satisfaction ramp


NAVY   = "#173A67"
BLUE   = "#1769C2"
TEAL   = "#159A8C"
ORANGE = "#E8833A"
RED    = "#C0392B"
GREY   = "#A7B0BA"
MUTED  = "#64748B"
TEXT   = "#334155"
WHITE  = "#FFFFFF"

bar_colors = [RED, ORANGE, GREY, BLUE, NAVY]   # 1 -> 5 stars



# 3. Summary statistics

mean_score    = (df["review_score"] * df["reviews"]).sum() / df["reviews"].sum()
total_reviews = int(df["reviews"].sum())
positive_pct  = df.loc[df["review_score"] >= 4, "pct"].sum()
negative_pct  = df.loc[df["review_score"] <= 2, "pct"].sum()
one_star_pct  = df.loc[df["review_score"] == 1, "pct"].iloc[0]


# 4. Figure
#  Do NOT use the save() helper — bbox_inches="tight" re-crops the
#  canvas and breaks the figure-fraction KPI cards and takeaway box.
#  subplots_adjust is used instead of ax.set_position() so the axes
#  stay tied to the same coordinate system as everything else.

FIG_W, FIG_H = 11, 7.0
AR = FIG_W / FIG_H          # keeps figure-fraction circles circular

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H), dpi=160)
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

fig.subplots_adjust(left=0.095, right=0.965, top=0.700, bottom=0.235)


# 5. Title and subtitle

fig.text(0.095, 0.955,
         "Ratings are polarised: mostly 5s, with a hard core of 1s",
         ha="left", va="center", fontsize=20, fontweight="bold", color=NAVY)

fig.text(0.095, 0.910,
         "Distribution of customer review scores across all reviewed orders",
         ha="left", va="center", fontsize=11.5, fontstyle="italic", color=MUTED)


# 6. KPI strip — four evenly spaced cards


kpis = [
    ("MEAN SCORE",     f"{mean_score:.2f} / 5",  NAVY),
    ("TOTAL REVIEWS",  f"{total_reviews:,}",     NAVY),
    ("RATED 4-5 STAR", f"{positive_pct:.1f}%",   BLUE),
    ("RATED 1-2 STAR", f"{negative_pct:.1f}%",   RED),
]

kx0, kw, ky, kh = 0.095, 0.2175, 0.760, 0.098

for i, (label, value, colour) in enumerate(kpis):
    x0 = kx0 + i * kw
    fig.patches.append(
        FancyBboxPatch((x0 + 0.006, ky), kw - 0.012, kh,
                       boxstyle="round,pad=0.006,rounding_size=0.010",
                       transform=fig.transFigure,
                       facecolor="#F6F8FB", edgecolor="#DCE3EC",
                       linewidth=1.0, zorder=5)
    )
    fig.text(x0 + kw/2, ky + kh - 0.028, label,
             ha="center", va="center", fontsize=8.5,
             color=MUTED, fontweight="bold", zorder=6)
    fig.text(x0 + kw/2, ky + 0.030, value,
             ha="center", va="center", fontsize=16,
             fontweight="bold", color=colour, zorder=6)


# 7. Bars

x = np.arange(len(df))
bars = ax.bar(x, df["reviews"], width=0.56, color=bar_colors,
              edgecolor="none", zorder=3)

ax.set_ylim(0, df["reviews"].max() * 1.30)


# 8. Labels — percentage above, count below it
# Two clearly separated offsets; a combined "x%\n(n)" string
#  crowds the top of the tall bar.


for bar, pct, n in zip(bars, df["pct"], df["reviews"]):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + df["reviews"].max() * 0.062,
            f"{pct:.1f}%", ha="center", va="bottom",
            fontsize=13, fontweight="bold", color=NAVY)
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + df["reviews"].max() * 0.018,
            f"{int(n):,}", ha="center", va="bottom",
            fontsize=9.5, color=MUTED)


# 9. Axes, grid, spines

ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda v, _: f"{v/1000:.0f}k" if v >= 1000 else f"{v:.0f}")
)
ax.set_ylabel("Number of reviews", fontsize=12, fontweight="bold",
              color=NAVY, labelpad=10)

ax.set_xticks(x)
ax.set_xticklabels([f"{int(v)}" for v in df["review_score"]],
                   fontsize=13, fontweight="bold", color=TEXT)
ax.set_xlabel("Review score (1 = worst, 5 = best)",
              fontsize=12, fontweight="bold", color=NAVY, labelpad=10)

ax.grid(axis="y", color="#D9E2EC", linestyle="--", linewidth=0.8, alpha=0.85, zorder=0)
ax.grid(axis="x", visible=False)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#AAB4C0")
ax.spines["bottom"].set_color("#AAB4C0")

ax.tick_params(axis="y", colors="#475569", labelsize=10, length=0)
ax.tick_params(axis="x", length=0, pad=6)


# 10. Key takeaway box

box_x, box_y = 0.095, 0.030
box_w, box_h = 0.870, 0.108

fig.patches.append(
    FancyBboxPatch((box_x, box_y), box_w, box_h,
                   boxstyle="round,pad=0.010,rounding_size=0.012",
                   transform=fig.transFigure,
                   facecolor="#EAF6F4", edgecolor=TEAL,
                   linewidth=1.3, zorder=10)
)

tcy = box_y + box_h / 2

fig.patches.append(
    Ellipse((0.132, tcy), width=0.026, height=0.026 * AR,
            transform=fig.transFigure, facecolor=WHITE,
            edgecolor=TEAL, linewidth=1.7, zorder=11)
)

for dx, h in [(-0.0062, 0.016), (0.0, 0.026), (0.0062, 0.037)]:
    fig.patches.append(
        Rectangle((0.132 + dx - 0.0022, tcy - 0.021), 0.0044, h,
                  transform=fig.transFigure, facecolor=TEAL,
                  edgecolor="none", zorder=12)
    )

fig.text(0.166, tcy, "Key takeaway:",
         fontsize=11.5, fontweight="bold", color="#0F766E", va="center", zorder=12)

fig.text(0.269, tcy + 0.024,
         f"{positive_pct:.1f}% of reviews are 4-5 star, but {one_star_pct:.1f}% are 1-star - the middle is thin.",
         fontsize=11.5, color=TEXT, va="center", zorder=12)

fig.text(0.269, tcy - 0.024,
         "Scores are ordinal and bimodal, so group comparisons use non-parametric tests, not ANOVA.",
         fontsize=11.5, color=TEXT, va="center", zorder=12)


# 11. Save

plt.savefig(f"{FIGDIR}/fig_18_review_distribution.png", dpi=300, facecolor="white")
plt.savefig(f"{FIGDIR}/fig_18_review_distribution.pdf",           facecolor="white")

plt.show()

### **Observation**
Review scores are strongly polarised: a large concentration of **5-star ratings**, a distinct cluster at **1 star**, and relatively few observations in the middle of the scale.

### **Methodological Importance**
Review score is **ordinal and bimodal**, rather than a continuous, normally distributed measure. Accordingly, the statistical analysis does not rely on ANOVA, which assumes interval-scale outcomes and stronger distributional conditions. Group comparisons use the **Kruskal–Wallis test**, followed by **Dunn's post-hoc tests with Holm adjustment** where appropriate. For regression, the outcome is also transformed into a binary **`poor_review`** indicator, defined as a review score of **≤ 2**.

The principle is simple: **choose the statistical method to fit the structure of the data, rather than applying a conventional test by default.**

## **Figure 9 — Delivery Lateness and Review Outcomes**

**The central exhibit of the report.**

**Question.** How strongly is delivery lateness associated with customer dissatisfaction?

**Grain.** One row per order. This figure must be generated from `v_fact_orders`, not an item-level view. At item grain, a six-item order would contribute its single review score six times, artificially inflating the sample size and understating uncertainty.

**Metric.** Lateness is defined as:

$$
\text{Lateness}
=
\text{Actual Delivery Date}
-
\text{Promised Delivery Date}
$$

Orders are grouped into five delivery-performance bands: **Early, On time, 1–3 days late, 4–7 days late, and 8+ days late**.

**Outcomes.** Each band reports both **average review score** and the **share of orders receiving a poor review (1–2 stars)**, allowing the relationship between fulfilment performance and customer dissatisfaction to be viewed from both the mean and the lower tail of the review distribution.

In [ ]:

# FIG 9 — LATENESS vs CUSTOMER SATISFACTION  (centrepiece)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch, Ellipse
from matplotlib.lines import Line2D


plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "DejaVu Sans"


# 1. Query
# GRAIN NOTE: this reads from v_orders_enriched joined to reviews,
# which is order grain — correct. Do NOT source this figure from
# v_fact_order_items: at item grain a six-item order contributes its
# single review score six times.

df = q("""
    WITH base AS (
        SELECT
            DATE_DIFF(DATE(o.delivered_ts), DATE(o.estimated_ts), DAY) AS gap,
            r.review_score
        FROM `olist.v_orders_enriched` o
        JOIN `olist.v_reviews_clean` r USING (order_id)
        WHERE o.order_status = 'delivered'
          AND o.delivered_ts IS NOT NULL
          AND o.estimated_ts IS NOT NULL
    )
    SELECT
        CASE WHEN gap <= -1           THEN 'Early'
             WHEN gap =  0            THEN 'On time'
             WHEN gap BETWEEN 1 AND 3 THEN '1-3d late'
             WHEN gap BETWEEN 4 AND 7 THEN '4-7d late'
             ELSE '8d+ late' END AS bucket,
        MIN(gap)                                                AS srt,
        COUNT(*)                                                AS orders,
        ROUND(AVG(review_score), 3)                             AS avg_score,
        ROUND(100.0 * COUNTIF(review_score <= 2) / COUNT(*), 1) AS pct_1_2_star
    FROM base
    GROUP BY bucket
    ORDER BY srt
""")

for col in ["avg_score", "pct_1_2_star", "orders"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["share"] = 100 * df["orders"] / df["orders"].sum()


# 2. Palette — a severity ramp, not one highlighted bar


NAVY     = "#244A78"
NAVY_L   = "#5B87BE"
ORANGE   = "#E8892F"
ORANGE_D = "#B4651C"     # darker: readable as text/tick colour
RED      = "#C0392B"
TEXT     = "#172B4D"
MUTED    = "#6F7D91"
GRID     = "#DCE3EB"
WHITE    = "#FFFFFF"


# 3. Figure
# Sized up: this is the report's key exhibit.
# NEVER save this one with bbox_inches="tight" — it re-crops the canvas
# and pulls the figure-fraction takeaway box out of alignment.

FIG_W, FIG_H = 12, 7.4

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H), dpi=160)
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

fig.subplots_adjust(left=0.085, right=0.900, top=0.735, bottom=0.290)

ax2 = ax.twinx()
x   = np.arange(len(df))

# 4. Title and subtitle


fig.text(0.085, 0.955,
         "Late delivery is strongly associated with poor reviews",
         ha="left", va="center", fontsize=21, fontweight="bold", color=TEXT)

fig.text(0.085, 0.908,
         "Customer ratings deteriorate sharply as delivery moves further beyond the promised date",
         ha="left", va="center", fontsize=11.5, color=MUTED, style="italic")


# 5. Bars — average review score


colors = [NAVY, NAVY, NAVY_L, ORANGE, RED]

bars = ax.bar(x, df["avg_score"], width=0.56, color=colors, zorder=3)
ax.set_ylim(0, 5.9)

for bar, value in zip(bars, df["avg_score"]):
    ax.text(bar.get_x() + bar.get_width()/2, value + 0.10,
            f"{value:.2f}", ha="center", va="bottom",
            fontsize=12, fontweight="bold", color=TEXT, zorder=5)


# 6. Line — share rated 1-2 stars

ax2.plot(x, df["pct_1_2_star"], color=ORANGE, linewidth=2.9,
         marker="o", markersize=8.5, markerfacecolor="white",
         markeredgecolor=ORANGE, markeredgewidth=2.2, zorder=6)

ax2.set_ylim(0, df["pct_1_2_star"].max() * 1.42)

offset = df["pct_1_2_star"].max() * 0.055

for xi, value in zip(x, df["pct_1_2_star"]):
    ax2.text(xi, value + offset, f"{value:.0f}%",
             ha="center", va="bottom", fontsize=11.5, fontweight="bold",
             color=ORANGE_D, zorder=7,
             bbox=dict(boxstyle="round,pad=0.2", facecolor=WHITE,
                       edgecolor="none", alpha=0.88))


# 7. Axes, grid, spines

ax.set_ylabel("Average review score", fontsize=12, fontweight="bold",
              color=NAVY, labelpad=10)
ax2.set_ylabel("Orders rated 1-2 stars (%)", fontsize=12, fontweight="bold",
               color=ORANGE_D, labelpad=12)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))

ax.set_xticks(x)
ax.set_xticklabels(df["bucket"], fontsize=11.5, fontweight="bold", color=TEXT)

ax.tick_params(axis="x", length=0, pad=8)
ax.tick_params(axis="y", colors=NAVY, length=0, labelsize=10)
ax2.tick_params(axis="y", colors=ORANGE_D, length=0, labelsize=10)

ax.grid(axis="y", linestyle="--", linewidth=0.7, color=GRID, alpha=0.85)
ax.grid(axis="x", visible=False)
ax2.grid(False)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax2.spines["top"].set_visible(False)
ax2.spines["left"].set_visible(False)
ax.spines["left"].set_color("#C9D1DC")
ax.spines["bottom"].set_color("#C9D1DC")
ax2.spines["right"].set_color(ORANGE)


# 8. Sample sizes AND share — 92% of orders sit in one band,
# so the count alone is misleading without the proportion


for i, row in df.iterrows():
    ax.text(i, -0.085, f"{int(row['orders']):,} orders",
            transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=9.5, color=MUTED)
    ax.text(i, -0.140, f"({row['share']:.1f}% of all)",
            transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=8.5, color=MUTED, fontstyle="italic")


# 9. Headline callout — the ratio, not the raw percentage

early = df["pct_1_2_star"].iloc[0]
late  = df["pct_1_2_star"].iloc[-1]

ax2.annotate(f"{late/early:.1f}x more 1-2 star reviews\nthan orders arriving early",
             xy=(x[-1], late),
             xytext=(2.05, df["pct_1_2_star"].max() * 1.22),
             fontsize=11.5, fontweight="bold", color=RED,
             ha="center", va="center",
             bbox=dict(boxstyle="round,pad=0.5", facecolor="#FDECEA",
                       edgecolor=RED, linewidth=1.4),
             arrowprops=dict(arrowstyle="-|>", color=RED, lw=1.8,
                             shrinkA=5, shrinkB=9,
                             connectionstyle="arc3,rad=-0.18"),
             zorder=10)


# 10. Legend — axis stated explicitly

ax.legend(
    [Rectangle((0, 0), 1, 1, color=NAVY),
     Line2D([0], [0], color=ORANGE, linewidth=2.6, marker="o", markersize=7.5,
            markerfacecolor="white", markeredgecolor=ORANGE)],
    ["Average review score (left axis)", "Share rated 1-2 stars (right axis)"],
    loc="upper left", frameon=False, fontsize=10
)

fig.text(0.492, 0.168, "Delivery performance vs promised date",
         ha="center", va="center", fontsize=12, fontweight="bold", color=TEXT)


# 11. Key takeaway box

box_x, box_y = 0.085, 0.028
box_w, box_h = 0.870, 0.110

fig.patches.append(
    FancyBboxPatch((box_x, box_y), box_w, box_h,
                   boxstyle="round,pad=0.010,rounding_size=0.012",
                   transform=fig.transFigure,
                   facecolor="#FDECEA", edgecolor=RED, linewidth=1.3, zorder=10)
)

tcy = box_y + box_h / 2
AR  = FIG_W / FIG_H

fig.patches.append(
    Ellipse((0.122, tcy), width=0.026, height=0.026 * AR,
            transform=fig.transFigure, facecolor=WHITE,
            edgecolor=RED, linewidth=1.7, zorder=11)
)

for dx, h in [(-0.0062, 0.014), (0.0, 0.025), (0.0062, 0.037)]:
    fig.patches.append(
        Rectangle((0.122 + dx - 0.0022, tcy - 0.020), 0.0044, h,
                  transform=fig.transFigure, facecolor=RED,
                  edgecolor="none", zorder=12)
    )

fig.text(0.156, tcy, "Key takeaway:",
         fontsize=11.5, fontweight="bold", color=RED, va="center", zorder=12)

fig.text(0.259, tcy + 0.025,
         f"Average rating falls from {df['avg_score'].iloc[0]:.2f} to "
         f"{df['avg_score'].iloc[-1]:.2f} once delivery slips 8+ days past the promise.",
         fontsize=11.5, color=TEXT, va="center", zorder=12)

fig.text(0.259, tcy - 0.025,
         "Association only: a failed order is both late and badly reviewed, so causality cannot be inferred.",
         fontsize=11.5, color=TEXT, va="center", zorder=12)


# 12. Save — no bbox_inches

plt.savefig(f"{FIGDIR}/fig_19_lateness_vs_review.png", dpi=300, facecolor="white")
plt.savefig(f"{FIGDIR}/fig_19_lateness_vs_review.pdf",           facecolor="white")

plt.show()

## **Observation**

The relationship is large and monotonic: as delivery lateness increases, both average review score and the share of poor reviews deteriorate sharply.

| Delivery band | Orders | Share | Avg. score | 1–2★ reviews |
|---|---:|---:|---:|---:|
| Early | 88,163 | 92.0% | 4.29 | 9.2% |
| On time | 1,280 | 1.3% | 4.03 | 12.4% |
| 1–3 days late | 1,852 | 1.9% | 3.29 | 32.1% |
| 4–7 days late | 1,748 | 1.8% | 2.10 | 67.6% |
| 8+ days late | 2,781 | 2.9% | 1.70 | 79.3% |

Average review score falls from **4.29 to 1.70** across the extreme bands. The share of 1–2 star reviews rises from **9.2% to 79.3%**, an **8.6× increase**. Compared with on-time orders, the share of poor reviews is approximately **6.4× higher across orders classified as late**. The previously reported χ² = 14,759, p < 0.001 was calculated on the **item-grain analysis** and therefore should **not** be presented as the final statistical result; the significance test must be re-run at the validated order grain.

## **Interpretation caveats**

**1. "Late" is measured against a heavily padded promise.** Approximately 92% of orders arrive before the estimated delivery date. An order classified as **8+ days late** has therefore exceeded an already-generous delivery window and may represent a **pathological fulfilment event**—such as a lost, damaged, or misrouted order—rather than simply a slow shipment.

**2. Reverse causality cannot be excluded.** A failed fulfilment event can simultaneously produce both a late delivery and a poor review. Lateness may therefore be a **marker of a broader fulfilment failure**, rather than the sole mechanism driving dissatisfaction. Because this is observational data, controls cannot establish causality; a randomized intervention would be required for that.

**3. The groups are highly unbalanced.** Approximately **92% of orders are early**, while each late band contains between 1,748 and 2,781 orders. Sample sizes are therefore shown explicitly so that the magnitude of the observed differences is not interpreted without their underlying group sizes.

## **Implication**

Delivery reliability has a **strong observed association with customer satisfaction**, and the approximately **6,400 orders across the three late bands** identify a relatively contained operational population for targeted intervention. However, whether improving delivery performance would increase **repeat purchasing** is a separate question; the later retention analysis does not support assuming that relationship.

---# Appendix · Supporting analysesRetained for completeness. These are context rather than findings, and are not intendedfor the main report.

### A1 · Payment Mix and Installments

Credit cards account for **75.4% of orders**, while *boleto* accounts for **19.9%**, reflecting the distinctive payment mix in the Brazilian marketplace. Average order value also tends to be higher among orders paid in more installments, rising from **R121 at one installment to R419 at ten installments**.

However, the relationship is **not consistently monotonic** at higher installment counts: AOV moves from **R310 at 8 installments → R205 at 9 → R419 at 10 → R125 at 11 → R377 at 12**. The 11-installment group contains only **23 orders**, making these extreme values sensitive to small sample sizes.

The most defensible pattern is therefore the **well-sampled increase from 1 to 6 installments, where AOV rises from R121 to R212**.

**Interpretation:** larger orders are **associated with** greater use of installments. The data does not establish that choosing more installments causes customers to spend more; customers making larger purchases may simply be more likely to choose financing.

In [ ]:
# ============================================================
# FIG 12 — PAYMENT MIX AND INSTALLMENTS vs AOV
# Presentation / Report Quality
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from matplotlib.patches import (
    FancyBboxPatch,
    Ellipse,
    Rectangle
)

from matplotlib.lines import Line2D


# ============================================================
# Global Plot Settings
# ============================================================

plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "DejaVu Sans"


# ============================================================
# 1. Queries
# ============================================================

mix = q("""
    SELECT
        primary_payment_type,
        COUNT(*) AS orders,
        ROUND(AVG(order_payment_total), 2) AS aov
    FROM `olist.v_payments_order`
    GROUP BY primary_payment_type
    ORDER BY orders DESC
""")

ins = q("""
    SELECT
        max_installments AS installments,
        COUNT(*) AS orders,
        ROUND(AVG(order_payment_total), 2) AS aov
    FROM `olist.v_payments_order`
    GROUP BY installments
    ORDER BY installments
""")

ins = ins[
    (ins["installments"] >= 1)
    & (ins["installments"] <= 12)
].copy()


# Drop the 3 'not_defined' rows — they'd render as
# an invisible bar with a label
mix = mix[
    mix["orders"] >= 10
].reset_index(drop=True)

total_orders = mix["orders"].sum()

mix["share"] = (
    100 * mix["orders"] / total_orders
)

mix["label"] = (
    mix["primary_payment_type"]
    .str.replace("_", " ")
    .str.title()
)


# ============================================================
# 2. Palette
# ============================================================

C_NAVY  = "#173F73"
C_TEAL  = "#19A68C"
C_OR    = "#E8833A"
C_OR_D  = "#C26428"
C_TEXT  = "#172033"
C_MUTED = "#667085"
C_GRID  = "#D9E1EA"
C_LIGHT = "#DCE5EF"
WHITE   = "#FFFFFF"


# ============================================================
# 3. Figure
#
# Do NOT use the save() helper — its bbox_inches="tight"
# re-crops the canvas and pushes the takeaway text off
# the figure entirely.
# ============================================================

FIG_W, FIG_H = 13, 6.9

fig, axes = plt.subplots(
    1,
    2,
    figsize=(FIG_W, FIG_H),
    dpi=160
)

fig.patch.set_facecolor("white")

for a in axes:
    a.set_facecolor("white")

fig.subplots_adjust(
    left=0.070,
    right=0.935,
    top=0.755,
    bottom=0.235,
    wspace=0.30
)


# ============================================================
# 4. Title and Subtitle
# ============================================================

fig.text(
    0.070,
    0.958,
    "Cards dominate payment mix, while larger baskets use more financing",
    ha="left",
    va="center",
    fontsize=20,
    fontweight="bold",
    color=C_TEXT
)

fig.text(
    0.070,
    0.912,
    "Payment-method mix and the relationship between installments, "
    "order volume and average order value",
    ha="left",
    va="center",
    fontsize=11.5,
    color=C_MUTED,
    style="italic"
)


# ============================================================
# 5. LEFT — Payment Method Mix
# ============================================================

ax = axes[0]

colors = [
    C_TEAL if i == 0 else C_NAVY
    for i in range(len(mix))
]

bars = ax.bar(
    mix["label"],
    mix["orders"],
    color=colors,
    width=0.62,
    edgecolor="white",
    linewidth=1.2,
    zorder=3
)

ax.set_ylim(
    0,
    mix["orders"].max() * 1.22
)


# ------------------------------------------------------------
# Labels Above Bars
# ------------------------------------------------------------

for bar, orders, share in zip(
    bars,
    mix["orders"],
    mix["share"]
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        orders + mix["orders"].max() * 0.02,
        f"{int(orders):,}\n({share:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
        color=(
            C_TEAL
            if orders == mix["orders"].max()
            else C_TEXT
        ),
        linespacing=1.3
    )


# ------------------------------------------------------------
# Axis Formatting
# ------------------------------------------------------------

ax.set_title(
    "Credit cards dominate the payment mix",
    fontsize=14.5,
    fontweight="bold",
    color=C_TEXT,
    pad=12
)

ax.set_ylabel(
    "Orders",
    fontsize=11,
    fontweight="bold",
    color=C_TEXT
)

ax.set_xlabel(
    "Payment method",
    fontsize=11,
    fontweight="bold",
    color=C_TEXT,
    labelpad=8
)

ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(
        lambda x, _: f"{x / 1000:.0f}k"
    )
)

ax.tick_params(
    axis="x",
    labelsize=10.5,
    colors=C_TEXT,
    length=0
)

ax.tick_params(
    axis="y",
    labelsize=9.5,
    colors=C_MUTED,
    length=0
)

ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.8,
    color=C_GRID,
    alpha=0.9,
    zorder=0
)

ax.grid(
    axis="x",
    visible=False
)

ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_color("#C5CED8")
ax.spines["bottom"].set_color("#C5CED8")


# ============================================================
# 6. RIGHT — Installments vs AOV
# ============================================================

ax = axes[1]
ax2 = ax.twinx()


# ------------------------------------------------------------
# Orders — Left Axis
# ------------------------------------------------------------

ax.bar(
    ins["installments"],
    ins["orders"],
    width=0.62,
    color=C_LIGHT,
    edgecolor="none",
    zorder=1
)


# ------------------------------------------------------------
# AOV — Right Axis
# ------------------------------------------------------------

ax2.plot(
    ins["installments"],
    ins["aov"],
    color=C_OR,
    linewidth=2.8,
    marker="o",
    markersize=6.5,
    markerfacecolor="white",
    markeredgecolor=C_OR,
    markeredgewidth=2,
    zorder=5
)


# ------------------------------------------------------------
# Axis Limits
# ------------------------------------------------------------

ax.set_ylim(
    0,
    ins["orders"].max() * 1.30
)

ax2.set_ylim(
    0,
    ins["aov"].max() * 1.32
)


# ------------------------------------------------------------
# Peak AOV Annotation
# ------------------------------------------------------------

peak = ins.loc[
    ins["aov"].idxmax()
]

ax2.scatter(
    peak["installments"],
    peak["aov"],
    s=130,
    color=C_OR,
    edgecolor="white",
    linewidth=2,
    zorder=6
)

ax2.annotate(
    f"Peak AOV  R\\$ {peak['aov']:,.0f}\n"
    f"at {int(peak['installments'])} installments",
    xy=(
        peak["installments"],
        peak["aov"]
    ),
    xytext=(
        peak["installments"] - 4.0,
        peak["aov"] * 1.10
    ),
    fontsize=10,
    fontweight="bold",
    color=C_OR_D,
    ha="center",
    va="center",
    bbox=dict(
        boxstyle="round,pad=0.45",
        facecolor="#FFF7ED",
        edgecolor=C_OR,
        linewidth=1.3
    ),
    arrowprops=dict(
        arrowstyle="-|>",
        color=C_OR,
        lw=1.5,
        shrinkA=4,
        shrinkB=8
    ),
    zorder=10
)


# ------------------------------------------------------------
# Titles and Labels
# ------------------------------------------------------------

ax.set_title(
    "Larger baskets are financed over more installments",
    fontsize=14.5,
    fontweight="bold",
    color=C_TEXT,
    pad=12
)

ax.set_xlabel(
    "Number of installments",
    fontsize=11,
    fontweight="bold",
    color=C_TEXT,
    labelpad=8
)

ax.set_ylabel(
    "Orders",
    fontsize=11,
    fontweight="bold",
    color=C_TEXT
)

ax2.set_ylabel(
    "Average order value (R\\$)",
    fontsize=11,
    fontweight="bold",
    color=C_OR_D,
    labelpad=10
)


# ------------------------------------------------------------
# Axis Formatting
# ------------------------------------------------------------

ax.set_xticks(
    range(1, 13)
)

ax.set_xlim(
    0.3,
    12.7
)

ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(
        lambda x, _: f"{x / 1000:.0f}k"
    )
)

ax2.yaxis.set_major_formatter(
    mticker.FuncFormatter(
        lambda x, _: f"{x:,.0f}"
    )
)

ax.tick_params(
    axis="x",
    labelsize=10,
    colors=C_TEXT,
    length=0
)

ax.tick_params(
    axis="y",
    labelsize=9.5,
    colors=C_MUTED,
    length=0
)

ax2.tick_params(
    axis="y",
    labelsize=9.5,
    colors=C_OR_D,
    length=0
)

ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.8,
    color=C_GRID,
    alpha=0.9,
    zorder=0
)

ax.grid(
    axis="x",
    visible=False
)

ax2.grid(False)

ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_color("#C5CED8")
ax.spines["bottom"].set_color("#C5CED8")

ax2.spines["top"].set_visible(False)
ax2.spines["left"].set_visible(False)
ax2.spines["right"].set_color(C_OR)


# ------------------------------------------------------------
# Legend
# ------------------------------------------------------------

ax.legend(
    handles=[
        Rectangle(
            (0, 0),
            1,
            1,
            color=C_LIGHT
        ),
        Line2D(
            [0],
            [0],
            color=C_OR,
            lw=2.6,
            marker="o",
            markersize=6,
            markerfacecolor="white",
            markeredgecolor=C_OR
        )
    ],
    labels=[
        "Orders (left axis)",
        "Average order value (right axis)"
    ],
    loc="upper left",
    fontsize=9.5,
    frameon=False,
    handlelength=1.6
)


# ============================================================
# 7. Key Takeaway Box
# ============================================================

box_x, box_y = 0.070, 0.030
box_w, box_h = 0.865, 0.115

fig.patches.append(
    FancyBboxPatch(
        (box_x, box_y),
        box_w,
        box_h,
        boxstyle=(
            "round,pad=0.010,"
            "rounding_size=0.012"
        ),
        transform=fig.transFigure,
        facecolor="#F4F7FB",
        edgecolor=C_NAVY,
        linewidth=1.3,
        zorder=10
    )
)

tcy = (
    box_y
    + box_h / 2
)

AR = (
    FIG_W / FIG_H
)


# ------------------------------------------------------------
# Takeaway Icon
# ------------------------------------------------------------

fig.patches.append(
    Ellipse(
        (0.110, tcy),
        width=0.026,
        height=0.026 * AR,
        transform=fig.transFigure,
        facecolor=WHITE,
        edgecolor=C_NAVY,
        linewidth=1.7,
        zorder=11
    )
)

for dx, h in [
    (-0.0062, 0.016),
    (0.0, 0.026),
    (0.0062, 0.038)
]:
    fig.patches.append(
        Rectangle(
            (
                0.110 + dx - 0.0022,
                tcy - 0.021
            ),
            0.0044,
            h,
            transform=fig.transFigure,
            facecolor=C_NAVY,
            edgecolor="none",
            zorder=12
        )
    )


# ------------------------------------------------------------
# Takeaway Values
# ------------------------------------------------------------

aov_1 = ins.loc[
    ins["installments"] == 1,
    "aov"
].iloc[0]

aov_10 = ins.loc[
    ins["installments"] == 10,
    "aov"
].iloc[0]


# ------------------------------------------------------------
# Takeaway Text
# ------------------------------------------------------------

fig.text(
    0.142,
    tcy,
    "Key takeaway:",
    fontsize=11.5,
    fontweight="bold",
    color=C_NAVY,
    va="center",
    zorder=12
)

fig.text(
    0.243,
    tcy + 0.025,
    f"Credit cards carry {mix.iloc[0]['share']:.1f}% of orders "
    f"and boleto {mix.iloc[1]['share']:.1f}% - "
    "a distinctly Brazilian mix.",
    fontsize=11.5,
    color=C_TEXT,
    va="center",
    zorder=12
)

fig.text(
    0.243,
    tcy - 0.025,
    f"AOV rises from R\\$ {aov_1:,.0f} at one installment "
    f"to R\\$ {aov_10:,.0f} at ten: "
    "financing unlocks bigger baskets.",
    fontsize=11.5,
    color=C_TEXT,
    va="center",
    zorder=12
)


# ============================================================
# 8. Save
# ============================================================

plt.savefig(
    f"{FIGDIR}/fig_12_payments.png",
    dpi=300,
    facecolor="white"
)

plt.savefig(
    f"{FIGDIR}/fig_12_payments.pdf",
    facecolor="white"
)

plt.show()

### **A2 · Order Status Distribution**

**97.02% of orders reach the customer**, while cancelled and unavailable orders together account for just **1.24%** of all orders.

This provides a useful data-quality and business-context check: **order failure is unlikely to explain the observed retention gap**. Since the finding does not materially drive the main analytical story, it is retained as an **appendix item rather than a main-report figure**.

In [ ]:
# ============================================================
# FIG 02 — ORDER STATUS DISTRIBUTION
# 4:3 | Presentation / Report Quality
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.patches import (
    FancyBboxPatch,
    Circle
)

from matplotlib.lines import Line2D


# ============================================================
# 0. Disable LaTeX, Set a Consistent Font
# ============================================================

plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "DejaVu Sans"


# ============================================================
# 1. Query Order Status Data
# ============================================================

df = q("""
    SELECT
        order_status,
        COUNT(*) AS orders,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS pct
    FROM `olist.v_orders_enriched`
    GROUP BY order_status
    ORDER BY orders DESC
""")

df = (
    df
    .sort_values(
        "orders",
        ascending=False
    )
    .reset_index(drop=True)
)


# ============================================================
# 2. Colour Palette
# ============================================================

BLUE       = "#1557C0"
DARK_BLUE  = "#0B3D91"
LIGHT_BLUE = "#EEF5FF"
ORANGE     = "#F59E0B"
BLACK      = "#151515"
GREY       = "#555555"
GRID       = "#D9DEE7"
WHITE      = "#FFFFFF"


# ============================================================
# 3. Figure — Exact 4:3
#
# NOTE:
# Never save with bbox_inches="tight" here. It re-crops
# the canvas and knocks every figure-fraction element
# (KPI card, takeaway box, icons) out of alignment
# with the axes.
# ============================================================

fig, ax = plt.subplots(
    figsize=(12, 9),
    dpi=150
)

fig.subplots_adjust(
    left=0.155,     # room for the longest status label
    right=0.955,
    top=0.775,
    bottom=0.255
)


# ============================================================
# 4. Title and Subtitle
# ============================================================

fig.text(
    0.50,
    0.950,
    "Nearly all orders deliver; only a small tail fails",
    ha="center",
    va="center",
    fontsize=25,
    fontweight="bold",
    color=BLACK
)

fig.text(
    0.50,
    0.908,
    "Order-status distribution across the Olist marketplace",
    ha="center",
    va="center",
    fontsize=15,
    fontstyle="italic",
    color=GREY
)


# ============================================================
# 5. Prepare Bar Data
#    Reversed so the largest sits on top
# ============================================================

plot_df = df.iloc[::-1].copy()

# Prettify labels:
# "unavailable" -> "Unavailable"
plot_df["label"] = (
    plot_df["order_status"]
    .str.replace("_", " ")
    .str.title()
)

colors = [
    BLUE if status == "delivered" else ORANGE
    for status in plot_df["order_status"]
]


# ============================================================
# 6. Horizontal Bars
# ============================================================

bars = ax.barh(
    plot_df["label"],
    plot_df["orders"],
    color=colors,
    height=0.62,
    edgecolor="none",
    zorder=3
)


# ============================================================
# 7. Log Scale with Enough Headroom for Labels
# ============================================================

ax.set_xscale("log")

ax.set_xlim(
    0.7,
    df["orders"].max() * 18
)


# ============================================================
# 8. Data Labels — Absolute Count + Percentage
# ============================================================

for bar, pct, cnt, status in zip(
    bars,
    plot_df["pct"],
    plot_df["orders"],
    plot_df["order_status"]
):
    ax.text(
        bar.get_width() * 1.25,
        bar.get_y() + bar.get_height() / 2,
        f"{cnt:,}  ({pct:.2f}%)",
        ha="left",
        va="center",
        fontsize=11.5,
        fontweight="bold",
        color=(
            BLUE
            if status == "delivered"
            else BLACK
        ),
        zorder=5
    )


# ============================================================
# 9. Highlight the "Delivered" Tick Label
# ============================================================

for label in ax.get_yticklabels():
    label.set_fontweight("bold")

    label.set_color(
        DARK_BLUE
        if label.get_text() == "Delivered"
        else BLACK
    )


# ============================================================
# 10. Axis Labels and Ticks
# ============================================================

ax.set_xlabel(
    "Number of orders (log scale)",
    fontsize=14,
    fontweight="bold",
    labelpad=12,
    color=BLACK
)

ax.set_ylabel("")

ax.tick_params(
    axis="x",
    labelsize=11,
    length=5,
    width=1,
    colors=BLACK
)

ax.tick_params(
    axis="y",
    labelsize=13,
    length=0,
    pad=10
)


# ============================================================
# 11. Grid and Spines
# ============================================================

ax.grid(
    True,
    axis="x",
    linestyle="--",
    linewidth=0.8,
    color=GRID,
    alpha=0.85,
    zorder=0
)

ax.grid(
    False,
    axis="y"
)

ax.set_axisbelow(True)

for side in [
    "top",
    "right",
    "left",
    "bottom"
]:
    ax.spines[side].set_visible(True)
    ax.spines[side].set_linewidth(1.0)
    ax.spines[side].set_color("#222222")


# ============================================================
# 12. Separator Under the Delivered Bar
# ============================================================

delivered_index = list(
    plot_df["order_status"]
).index("delivered")

ax.axhline(
    delivered_index - 0.5,
    color=BLUE,
    linewidth=1.4,
    linestyle="--",
    alpha=0.9,
    zorder=4
)


# ============================================================
# 13. TOP-RIGHT KPI CARD
# ============================================================

delivered_pct = float(
    df.loc[
        df["order_status"] == "delivered",
        "pct"
    ].iloc[0]
)

card_x, card_y = 0.800, 0.800
card_w, card_h = 0.155, 0.115

fig.patches.append(
    FancyBboxPatch(
        (card_x, card_y),
        card_w,
        card_h,
        boxstyle=(
            "round,pad=0.008,"
            "rounding_size=0.015"
        ),
        transform=fig.transFigure,
        facecolor=LIGHT_BLUE,
        edgecolor="#8CB3E8",
        linewidth=1.5,
        zorder=10
    )
)


# ------------------------------------------------------------
# Tick Icon
# ------------------------------------------------------------

icon_x = card_x + 0.030
icon_y = card_y + card_h - 0.032

fig.patches.append(
    Circle(
        (icon_x, icon_y),
        0.019,
        transform=fig.transFigure,
        facecolor=WHITE,
        edgecolor=BLUE,
        linewidth=1.5,
        zorder=12
    )
)

fig.add_artist(
    Line2D(
        [
            icon_x - 0.008,
            icon_x - 0.002,
            icon_x + 0.009
        ],
        [
            icon_y,
            icon_y - 0.007,
            icon_y + 0.009
        ],
        transform=fig.transFigure,
        color=BLUE,
        linewidth=2.2,
        solid_capstyle="round",
        zorder=13
    )
)

fig.text(
    card_x + 0.105,
    icon_y,
    f"{delivered_pct:.2f}%",
    ha="center",
    va="center",
    fontsize=17,
    fontweight="bold",
    color=BLUE,
    zorder=13
)

fig.text(
    card_x + card_w / 2,
    card_y + 0.030,
    "of orders delivered",
    ha="center",
    va="center",
    fontsize=10.5,
    color=BLACK,
    zorder=13
)


# ============================================================
# 14. BOTTOM KEY TAKEAWAY BOX
# ============================================================

box_x, box_y = 0.085, 0.035
box_w, box_h = 0.870, 0.105

fig.patches.append(
    FancyBboxPatch(
        (box_x, box_y),
        box_w,
        box_h,
        boxstyle=(
            "round,pad=0.008,"
            "rounding_size=0.015"
        ),
        transform=fig.transFigure,
        facecolor=LIGHT_BLUE,
        edgecolor=BLUE,
        linewidth=1.7,
        zorder=2
    )
)


# Vertical centre — everything aligns off this
tcy = box_y + box_h / 2


# ------------------------------------------------------------
# Shopping-Bag Icon
# ------------------------------------------------------------

fig.patches.append(
    Circle(
        (0.125, tcy),
        0.022,
        transform=fig.transFigure,
        facecolor=WHITE,
        edgecolor=BLUE,
        linewidth=1.8,
        zorder=5
    )
)

fig.patches.append(
    FancyBboxPatch(
        (0.1175, tcy - 0.011),
        0.015,
        0.015,
        boxstyle="round,pad=0.002",
        transform=fig.transFigure,
        facecolor=BLUE,
        edgecolor=BLUE,
        linewidth=1,
        zorder=6
    )
)

fig.add_artist(
    Line2D(
        [
            0.1215,
            0.1215,
            0.1285,
            0.1285
        ],
        [
            tcy + 0.004,
            tcy + 0.010,
            tcy + 0.010,
            tcy + 0.004
        ],
        transform=fig.transFigure,
        color=BLUE,
        linewidth=1.6,
        solid_capstyle="round",
        zorder=7
    )
)


# ------------------------------------------------------------
# Takeaway Text
# ------------------------------------------------------------

fig.text(
    0.163,
    tcy,
    "Key takeaway:",
    ha="left",
    va="center",
    fontsize=13.5,
    fontweight="bold",
    color=DARK_BLUE,
    zorder=6
)

fig.text(
    0.283,
    tcy + 0.024,
    "97.02% of orders reach the customer; "
    "fulfilment reliability is not the problem.",
    ha="left",
    va="center",
    fontsize=12.5,
    color=BLACK,
    zorder=6
)

fig.text(
    0.283,
    tcy - 0.024,
    "Cancelled and unavailable orders together account "
    "for just 1.24% of volume.",
    ha="left",
    va="center",
    fontsize=12.5,
    color=BLACK,
    zorder=6
)


# ============================================================
# 15. Save — No bbox_inches
#     Manual layout is preserved
# ============================================================

plt.savefig(
    f"{FIGDIR}/fig_02_status_funnel.png",
    dpi=300,
    facecolor="white"
)

plt.savefig(
    f"{FIGDIR}/fig_02_status_funnel.pdf",
    facecolor="white"
)


# ============================================================
# 16. Display
# ============================================================

plt.show()

### **A3 · Demand Against Trailing Baseline**

Monthly order volume is compared against a **trailing three-month average**, with major Brazilian retail periods annotated to identify potential demand spikes.

**Why this is an appendix item.** The dataset covers **less than two years**, so most festivals and retail events occur only once in the observation window. A single occurrence can indicate an event effect, but it is not sufficient to establish a **repeatable seasonal pattern**. Black Friday 2017 is the only window showing a clear and distinct demand signal; therefore, the analysis does not support a broader seasonality claim.

In [ ]:
# ============================================================
# FIG 04 — MONTHLY VOLUME VS TRAILING BASELINE
# Presentation-style / 16:9 layout
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D


# ============================================================
# 0. Disable LaTeX
# ============================================================

plt.rcParams["text.usetex"] = False


# ============================================================
# 1. Query Monthly Order Data
# ============================================================

df = q("""
    WITH monthly AS (
        SELECT
            DATE_TRUNC(
                DATE(purchase_ts),
                MONTH
            ) AS m,
            COUNT(*) AS orders

        FROM `olist.v_orders_enriched`

        WHERE DATE(purchase_ts)
              BETWEEN '2017-01-01' AND '2018-08-31'

        GROUP BY m
    )

    SELECT
        m AS order_month,
        orders,

        ROUND(
            100.0 * (
                orders
                - AVG(orders) OVER (
                    ORDER BY m
                    ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
                )
            )
            / NULLIF(
                AVG(orders) OVER (
                    ORDER BY m
                    ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
                ),
                0
            ),
            1
        ) AS pct_vs_baseline

    FROM monthly

    ORDER BY m
""")


# ============================================================
# 2. Prepare Data
# ============================================================

df["order_month"] = pd.to_datetime(
    df["order_month"]
)

d = (
    df
    .dropna(
        subset=["pct_vs_baseline"]
    )
    .copy()
)

d = (
    d
    .sort_values("order_month")
    .reset_index(drop=True)
)


# ============================================================
# 3. Professional Colour Palette
# ============================================================

NAVY = "#172554"          # Main title / dark text
PURPLE = "#6D4BC3"        # Positive bars
PURPLE_DARK = "#4C2A9B"   # Annotation
RED = "#E85D5D"            # Negative bars
GRID = "#D9DDE7"          # Grid
BLACK = "#111827"
GREY = "#5B6472"
WHITE = "#FFFFFF"
LIGHT_PURPLE = "#F5F1FF"


# ============================================================
# 4. Create Figure
# ============================================================

fig, ax = plt.subplots(
    figsize=(14, 8),
    dpi=150
)

fig.subplots_adjust(
    left=0.085,
    right=0.965,
    top=0.78,
    bottom=0.24
)


# ============================================================
# 5. Main Title
# ============================================================

fig.text(
    0.5,
    0.945,
    "Demand lift against baseline — festival months stand out",
    ha="center",
    va="center",
    fontsize=27,
    fontweight="bold",
    color=NAVY
)


# ============================================================
# 6. Subtitle
# ============================================================

fig.text(
    0.5,
    0.905,
    "% change vs trailing 3-month average",
    ha="center",
    va="center",
    fontsize=17,
    fontstyle="italic",
    color=GREY
)


# ============================================================
# 7. Bar Colours
# ============================================================

bar_colors = [
    PURPLE if v >= 0 else RED
    for v in d["pct_vs_baseline"]
]


# ============================================================
# 8. Plot Bars
# ============================================================

bars = ax.bar(
    d["order_month"],
    d["pct_vs_baseline"],
    width=20,
    color=bar_colors,
    edgecolor="none",
    zorder=3
)


# ============================================================
# 9. Zero Baseline
# ============================================================

ax.axhline(
    0,
    color=BLACK,
    linewidth=1.2,
    zorder=4
)


# ============================================================
# 10. Add Percentage Labels
# ============================================================

for bar, value in zip(
    bars,
    d["pct_vs_baseline"]
):
    x = (
        bar.get_x()
        + bar.get_width() / 2
    )

    if value >= 0:
        ax.text(
            x,
            value + 3,
            f"{value:.0f}%",
            ha="center",
            va="bottom",
            fontsize=12,
            fontweight="bold",
            color=BLACK,
            zorder=5
        )

    else:
        ax.text(
            x,
            value - 3,
            f"{value:.0f}%",
            ha="center",
            va="top",
            fontsize=12,
            fontweight="bold",
            color=BLACK,
            zorder=5
        )


# ============================================================
# 11. Highlight November 2017 — Festive Season
# ============================================================

festive_month = pd.Timestamp(
    "2017-11-01"
)

festive_row = d[
    d["order_month"] == festive_month
]

if not festive_row.empty:

    festive_value = (
        festive_row[
            "pct_vs_baseline"
        ].iloc[0]
    )

    ax.annotate(
        "Festive season\n(Nov 2017)",
        xy=(
            festive_month,
            festive_value
        ),
        xytext=(
            festive_month
            + pd.DateOffset(months=1),
            108
        ),
        ha="center",
        va="center",
        fontsize=13,
        fontweight="bold",
        color=PURPLE_DARK,
        bbox=dict(
            boxstyle="round,pad=0.65",
            facecolor=WHITE,
            edgecolor="#8B5CF6",
            linewidth=1.8
        ),
        arrowprops=dict(
            arrowstyle="-|>",
            color="#8B5CF6",
            linewidth=1.8,
            shrinkA=5,
            shrinkB=8,
            connectionstyle="arc3,rad=0"
        ),
        zorder=10
    )


# ============================================================
# 12. Axis Labels
# ============================================================

ax.set_ylabel(
    "% vs trailing 3-month average",
    fontsize=15,
    fontweight="bold",
    color=NAVY,
    labelpad=12
)

ax.set_xlabel(
    "Month",
    fontsize=15,
    fontweight="bold",
    color=NAVY,
    labelpad=12
)


# ============================================================
# 13. Y-Axis Formatting
# ============================================================

ax.set_ylim(
    -20,
    140
)

ax.set_yticks(
    np.arange(
        -20,
        141,
        20
    )
)

ax.tick_params(
    axis="y",
    labelsize=12,
    colors=BLACK
)


# ============================================================
# 14. X-Axis Formatting
# ============================================================

ax.set_xlim(
    pd.Timestamp("2016-12-15"),
    pd.Timestamp("2018-08-25")
)

ax.xaxis.set_major_locator(
    mdates.MonthLocator(
        interval=2
    )
)

ax.xaxis.set_major_formatter(
    mdates.DateFormatter("%Y-%m")
)

plt.setp(
    ax.get_xticklabels(),
    rotation=30,
    ha="right",
    fontsize=12
)


# ============================================================
# 15. Grid
# ============================================================

ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.9,
    color=GRID,
    alpha=0.85,
    zorder=0
)

ax.grid(
    axis="x",
    visible=False
)

ax.set_axisbelow(True)


# ============================================================
# 16. Clean Borders
# ============================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_color("#333333")
ax.spines["bottom"].set_color("#333333")

ax.spines["left"].set_linewidth(1.1)
ax.spines["bottom"].set_linewidth(1.1)


# ============================================================
# 17. Key Takeaway Box
# ============================================================

box_x = 0.11
box_y = 0.055
box_width = 0.78
box_height = 0.095

takeaway_box = FancyBboxPatch(
    (box_x, box_y),
    box_width,
    box_height,
    boxstyle=(
        "round,pad=0.008,"
        "rounding_size=0.012"
    ),
    transform=fig.transFigure,
    facecolor=LIGHT_PURPLE,
    edgecolor="#7048C8",
    linewidth=1.8,
    zorder=2
)

fig.patches.append(
    takeaway_box
)


# ============================================================
# 18. Icon Circle
# ============================================================

circle = plt.Circle(
    (0.145, 0.102),
    0.022,
    transform=fig.transFigure,
    facecolor=WHITE,
    edgecolor="#7048C8",
    linewidth=1.8,
    zorder=5
)

fig.patches.append(
    circle
)


# ============================================================
# 19. Simple Upward-Trend Icon
# ============================================================

fig.add_artist(
    Line2D(
        [0.136, 0.142, 0.149, 0.157],
        [0.096, 0.101, 0.099, 0.110],
        transform=fig.transFigure,
        color="#7048C8",
        linewidth=2.5,
        zorder=6
    )
)


# ============================================================
# 20. Takeaway Text
# ============================================================

fig.text(
    0.21,
    0.107,
    "Key takeaway:",
    ha="left",
    va="center",
    fontsize=14,
    fontweight="bold",
    color="#4C2A9B"
)

fig.text(
    0.35,
    0.107,
    "Demand spikes in festival months, especially Nov 2017,",
    ha="left",
    va="center",
    fontsize=13.5,
    color=BLACK
)

fig.text(
    0.35,
    0.077,
    "while mid-2018 shows mild declines below baseline.",
    ha="left",
    va="center",
    fontsize=13.5,
    color=BLACK
)


# ============================================================
# 21. Save
# ============================================================

save(
    fig,
    "fig_04_festival_lift"
)


# Optional direct PNG/PDF save if needed:
#
# plt.savefig(
#     f"{FIGDIR}/fig_04_festival_lift.png",
#     dpi=300,
#     bbox_inches="tight",
#     facecolor="white"
# )


# ============================================================
# 22. Display
# ============================================================

plt.show()

### **A4 · Ordering Time-of-Day Pattern**

Demand is concentrated on weekdays, with **Monday the busiest at 16.2K orders**. Weekend volume is approximately **30% below the weekday average**, while orders remain relatively stable throughout the day, forming a broad plateau from **10:00 to 22:00** rather than distinct morning and evening peaks.

**Why this is an appendix item.** The pattern does not lead to a specific business decision in this analysis. Since Olist does not appear to manage fulfilment capacity against an hourly demand curve, the result is best treated as **descriptive context rather than an actionable product insight**.

In [ ]:
# ============================================================
# FIG 05 — DAY-OF-WEEK AND HOUR-OF-DAY
# Two-panel | Presentation / Report Quality
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from matplotlib.patches import FancyBboxPatch, Circle
from matplotlib.lines import Line2D

plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "DejaVu Sans"


# ------------------------------------------------------------
# 1. Queries
# ------------------------------------------------------------

dow = q("""
    SELECT
        EXTRACT(DAYOFWEEK FROM purchase_ts) AS dow,
        FORMAT_DATE('%a', DATE(purchase_ts)) AS weekday,
        COUNT(*) AS orders
    FROM `olist.v_orders_enriched`
    GROUP BY dow, weekday
    ORDER BY dow
""")

hod = q("""
    SELECT
        EXTRACT(HOUR FROM purchase_ts) AS hour,
        COUNT(*) AS orders
    FROM `olist.v_orders_enriched`
    GROUP BY hour
    ORDER BY hour
""")

weekday_order = [
    "Sun",
    "Mon",
    "Tue",
    "Wed",
    "Thu",
    "Fri",
    "Sat"
]

dow["weekday"] = pd.Categorical(
    dow["weekday"],
    categories=weekday_order,
    ordered=True
)

dow = (
    dow
    .sort_values("weekday")
    .reset_index(drop=True)
)

dow["orders"] = dow["orders"].astype(float)
hod["orders"] = hod["orders"].astype(float)


# ------------------------------------------------------------
# 2. Palette
# ------------------------------------------------------------

NAVY = "#172554"
BLUE = "#2563EB"
PURPLE = "#6D4BC3"
PURPLE_DARK = "#4C2A9B"
SLATE = "#94A3B8"
LIGHT_BLUE = "#EAF2FF"
GRID = "#D9DDE7"
BLACK = "#111827"
GREY = "#5B6472"
WHITE = "#FFFFFF"


# ------------------------------------------------------------
# 3. Figure
#
# Never save with bbox_inches="tight" — it re-crops the canvas
# and knocks the figure-fraction takeaway box out of alignment.
# ------------------------------------------------------------

fig, (ax1, ax2) = plt.subplots(
    1,
    2,
    figsize=(14, 7.5),
    dpi=150
)

fig.subplots_adjust(
    left=0.072,
    right=0.968,
    top=0.760,
    bottom=0.290,   # deep bottom margin: axis labels + legend + takeaway
    wspace=0.19
)


# ------------------------------------------------------------
# 4. Title and subtitle
# ------------------------------------------------------------

fig.text(
    0.5,
    0.945,
    "Weekday-heavy, with demand sustained from late morning to late evening",
    ha="center",
    va="center",
    fontsize=22,
    fontweight="bold",
    color=NAVY
)

fig.text(
    0.5,
    0.898,
    "Order volume distribution across day of week and hour of day",
    ha="center",
    va="center",
    fontsize=15,
    fontstyle="italic",
    color=GREY
)


# ============================================================
# 5. LEFT PANEL — day of week
# ============================================================

is_weekday = dow["weekday"].isin(
    ["Mon", "Tue", "Wed", "Thu", "Fri"]
)

colors = [
    BLUE if w else SLATE
    for w in is_weekday
]

colors[
    list(dow["weekday"]).index("Mon")
] = PURPLE  # busiest day

max_orders = dow["orders"].max()

bars = ax1.bar(
    dow["weekday"].astype(str),
    dow["orders"],
    color=colors,
    width=0.72,
    edgecolor="none",
    zorder=3
)

for bar, value in zip(bars, dow["orders"]):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        value + max_orders * 0.02,
        f"{value / 1000:.1f}k",
        ha="center",
        va="bottom",
        fontsize=11.5,
        fontweight="bold",
        color=BLACK
    )

ax1.set_ylim(
    0,
    max_orders * 1.16
)

ax1.set_title(
    "By day of week",
    fontsize=17,
    fontweight="bold",
    color=NAVY,
    pad=12
)

ax1.set_ylabel(
    "Orders",
    fontsize=13.5,
    fontweight="bold",
    color=NAVY,
    labelpad=10
)

ax1.set_xlabel(
    "Day of week",
    fontsize=13.5,
    fontweight="bold",
    color=NAVY,
    labelpad=10
)

ax1.yaxis.set_major_formatter(
    mticker.FuncFormatter(
        lambda x, _: f"{x / 1000:.0f}k"
    )
)

ax1.tick_params(
    axis="y",
    labelsize=11
)

ax1.tick_params(
    axis="x",
    labelsize=12,
    pad=6
)

ax1.grid(
    axis="y",
    linestyle="--",
    linewidth=0.9,
    color=GRID,
    alpha=0.85,
    zorder=0
)

ax1.grid(
    axis="x",
    visible=False
)

ax1.set_axisbelow(True)

ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

ax1.spines["left"].set_color("#333333")
ax1.spines["bottom"].set_color("#333333")

ax1.legend(
    handles=[
        plt.Rectangle(
            (0, 0),
            1,
            1,
            color=PURPLE
        ),
        plt.Rectangle(
            (0, 0),
            1,
            1,
            color=BLUE
        ),
        plt.Rectangle(
            (0, 0),
            1,
            1,
            color=SLATE
        )
    ],
    labels=[
        "Busiest day",
        "Weekday",
        "Weekend"
    ],
    fontsize=10,
    frameon=False,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.255),
    ncol=3
)


# ============================================================
# 6. RIGHT PANEL — hour of day
# ============================================================

ax2.plot(
    hod["hour"],
    hod["orders"],
    color=PURPLE,
    linewidth=2.8,
    marker="o",
    markersize=6,
    markerfacecolor=PURPLE,
    markeredgecolor=WHITE,
    markeredgewidth=1.4,
    zorder=4
)

ax2.fill_between(
    hod["hour"],
    hod["orders"],
    color=PURPLE,
    alpha=0.12,
    zorder=2
)

# ONE plateau band — the data shows a single sustained period,
# not a mid-morning peak plus a separate evening peak.
ax2.axvspan(
    10,
    22,
    color=PURPLE,
    alpha=0.08,
    zorder=1
)

plateau = hod["orders"][
    (hod["hour"] >= 10)
    & (hod["hour"] <= 22)
]

ax2.text(
    16,
    hod["orders"].max() * 1.10,
    f"Sustained plateau  10:00-22:00\n"
    f"{plateau.min() / 1000:.1f}k - "
    f"{plateau.max() / 1000:.1f}k orders/hour",
    ha="center",
    va="center",
    fontsize=10.5,
    fontweight="bold",
    color=PURPLE_DARK,
    bbox=dict(
        boxstyle="round,pad=0.45",
        facecolor=WHITE,
        edgecolor=PURPLE,
        linewidth=1.5
    ),
    zorder=10
)

trough = hod.loc[
    hod["orders"].idxmin()
]

ax2.annotate(
    f"Trough {int(trough.hour):02d}:00",
    xy=(
        trough.hour,
        trough.orders
    ),
    xytext=(
        trough.hour + 1.0,
        hod["orders"].max() * 0.52
    ),
    ha="center",
    va="center",
    fontsize=10,
    fontweight="bold",
    color=GREY,
    arrowprops=dict(
        arrowstyle="-|>",
        color=GREY,
        linewidth=1.4,
        shrinkA=4,
        shrinkB=6
    ),
    zorder=10
)

ax2.set_ylim(
    0,
    hod["orders"].max() * 1.26
)

ax2.set_xlim(
    -0.6,
    23.6
)

ax2.set_title(
    "By hour of day",
    fontsize=17,
    fontweight="bold",
    color=NAVY,
    pad=12
)

ax2.set_xlabel(
    "Hour of day",
    fontsize=13.5,
    fontweight="bold",
    color=NAVY,
    labelpad=10
)

ax2.set_ylabel(
    "Orders",
    fontsize=13.5,
    fontweight="bold",
    color=NAVY,
    labelpad=10
)

ax2.set_xticks(
    np.arange(0, 24, 2)
)

ax2.set_xticklabels(
    [
        f"{h:02d}"
        for h in range(0, 24, 2)
    ],
    fontsize=11
)

ax2.yaxis.set_major_formatter(
    mticker.FuncFormatter(
        lambda x, _: f"{x / 1000:.0f}k"
    )
)

ax2.tick_params(
    axis="y",
    labelsize=11
)

ax2.grid(
    axis="y",
    linestyle="--",
    linewidth=0.9,
    color=GRID,
    alpha=0.85,
    zorder=0
)

ax2.grid(
    axis="x",
    visible=False
)

ax2.set_axisbelow(True)

ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

ax2.spines["left"].set_color("#333333")
ax2.spines["bottom"].set_color("#333333")


# ============================================================
# 7. Key takeaway box
# ============================================================

box_x = 0.072
box_y = 0.035
box_w = 0.896
box_h = 0.115

fig.patches.append(
    FancyBboxPatch(
        (box_x, box_y),
        box_w,
        box_h,
        boxstyle="round,pad=0.008,rounding_size=0.012",
        transform=fig.transFigure,
        facecolor=LIGHT_BLUE,
        edgecolor=BLUE,
        linewidth=1.8,
        zorder=2
    )
)

tcy = box_y + box_h / 2

fig.patches.append(
    Circle(
        (0.112, tcy),
        0.024,
        transform=fig.transFigure,
        facecolor=WHITE,
        edgecolor=BLUE,
        linewidth=1.8,
        zorder=5
    )
)

fig.add_artist(
    Line2D(
        [0.1015, 0.1085, 0.1155, 0.1225],
        [
            tcy - 0.008,
            tcy + 0.002,
            tcy - 0.004,
            tcy + 0.012
        ],
        transform=fig.transFigure,
        color=BLUE,
        linewidth=2.5,
        solid_capstyle="round",
        zorder=6
    )
)

fig.text(
    0.150,
    tcy,
    "Key takeaway:",
    ha="left",
    va="center",
    fontsize=14,
    fontweight="bold",
    color=NAVY,
    zorder=6
)

fig.text(
    0.268,
    tcy + 0.026,
    "Monday is the busiest day (16.2k); weekends run ~30% below the weekday average.",
    ha="left",
    va="center",
    fontsize=13,
    color=BLACK,
    zorder=6
)

fig.text(
    0.268,
    tcy - 0.026,
    "Hourly demand is a broad 10:00-22:00 plateau, not a morning/evening double peak.",
    ha="left",
    va="center",
    fontsize=13,
    color=BLACK,
    zorder=6
)


# ============================================================
# 8. Save — no bbox_inches, no save() helper (it uses tight)
# ============================================================

plt.savefig(
    f"{FIGDIR}/fig_05_dow_hour.png",
    dpi=300,
    facecolor="white"
)

plt.savefig(
    f"{FIGDIR}/fig_05_dow_hour.pdf",
    facecolor="white"
)

plt.show()